In [1]:
from pathlib import Path
import subprocess
import zipfile
import pandas as pd
from PIL import Image, ImageOps
from tqdm.auto import tqdm

# Mount Drive if needed.
from google.colab import drive
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

PROJECTS_DIR = Path("/content/drive/MyDrive/final_year_project")

# Find the Okra folder without assuming capitalization.
okra_matches = [
    path for path in PROJECTS_DIR.iterdir()
    if path.is_dir() and path.name.casefold() == "okra"
]
if not okra_matches:
    raise FileNotFoundError(
        f"Could not find an 'okra' folder inside {PROJECTS_DIR}. "
        f"Folders found: {[p.name for p in PROJECTS_DIR.iterdir() if p.is_dir()]}"
    )

OKRA_DIR = okra_matches[0]
EXPERIMENT_DIR = OKRA_DIR / "okra_model_experiment"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def find_images(folder):
    return [
        path for path in folder.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    ]

# Use an extracted dataset if images are already present.
existing_images = find_images(OKRA_DIR)

if existing_images:
    RAW_DIR = OKRA_DIR
    print(f"Using images already present in: {RAW_DIR}")
else:
    archives = [
        path for path in OKRA_DIR.iterdir()
        if path.is_file() and path.suffix.lower() in {".zip", ".rar"}
    ]
    if not archives:
        raise FileNotFoundError(
            f"No image files or ZIP/RAR archive found in {OKRA_DIR}. "
            f"Items here: {[p.name for p in OKRA_DIR.iterdir()]}"
        )

    archive_path = archives[0]
    RAW_DIR = Path("/content/okra_raw")
    RAW_DIR.mkdir(parents=True, exist_ok=True)

    if archive_path.suffix.lower() == ".zip":
        print("Extracting ZIP archive...")
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(RAW_DIR)
    else:
        print("Extracting RAR archive...")
        install = subprocess.run(
            ["apt-get", "-qq", "install", "-y", "unrar"],
            capture_output=True,
            text=True,
        )
        if install.returncode != 0:
            raise RuntimeError("Could not install unrar:\n" + install.stderr[-1500:])

        extraction = subprocess.run(
            ["unrar", "x", "-o-", str(archive_path), str(RAW_DIR) + "/"],
            capture_output=True,
            text=True,
        )
        if extraction.returncode != 0:
            raise RuntimeError("RAR extraction failed:\n" + extraction.stderr[-1500:])

print("\nDataset location:", RAW_DIR)
print("Top-level items:")
for path in sorted(RAW_DIR.iterdir()):
    print(f"  {path.name}{'/' if path.is_dir() else ''}")

image_paths = find_images(RAW_DIR)
if not image_paths:
    raise RuntimeError(f"No supported image files found under {RAW_DIR}")

# Build a manifest and verify each image can be opened.
rows = []
unreadable = []

for image_path in tqdm(image_paths, desc="Checking Okra images"):
    # The immediate parent folder is treated as the class folder.
    label = image_path.parent.name

    try:
        with Image.open(image_path) as source:
            image = ImageOps.exif_transpose(source)
            width, height = image.size
            image.verify()

        rows.append({
            "image_path": str(image_path),
            "label": label,
            "width": width,
            "height": height,
            "format": image_path.suffix.lower(),
        })
    except Exception as error:
        unreadable.append({
            "image_path": str(image_path),
            "error": str(error),
        })

manifest = pd.DataFrame(rows)
manifest_path = EXPERIMENT_DIR / "okra_initial_manifest.csv"
manifest.to_csv(manifest_path, index=False)

print(f"\nReadable images: {len(manifest)}")
print(f"Unreadable/corrupted images: {len(unreadable)}")
print("\nImages per class folder:")
display(manifest["label"].value_counts().rename("images").to_frame())

print("\nMost common image dimensions:")
display(
    manifest.groupby(["width", "height"])
    .size()
    .sort_values(ascending=False)
    .head(10)
    .rename("images")
    .reset_index()
)

if unreadable:
    unreadable_path = EXPERIMENT_DIR / "okra_unreadable_images.csv"
    pd.DataFrame(unreadable).to_csv(unreadable_path, index=False)
    print("\nUnreadable-image report:", unreadable_path)

print("\nInitial manifest saved to:", manifest_path)

Mounted at /content/drive
Extracting ZIP archive...

Dataset location: /content/okra_raw
Top-level items:
  Raw_Image_Dataset/


Checking Okra images:   0%|          | 0/4362 [00:00<?, ?it/s]


Readable images: 4362
Unreadable/corrupted images: 0

Images per class folder:


,images
label,
Yellow_Vein_Mosaic_Virus,1185
Healthy_Leaf,1142
Insects_Eaten_Leaf,744
Leaf_Curl,554
Leaf_Spot,480
Fruit_Rot,257



Most common image dimensions:


,width,height,images
0,300,300,4265
1,6000,4000,78
2,3489,3693,3
3,3710,3325,3
4,2096,2122,3
5,3191,3432,2
6,3229,2836,1
7,3068,3426,1
8,2906,3276,1
9,3316,3047,1



Initial manifest saved to: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_initial_manifest.csv


In [2]:
import hashlib
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
MANIFEST_PATH = EXPERIMENT_DIR / "okra_initial_manifest.csv"

manifest = pd.read_csv(MANIFEST_PATH)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

hashes = []
missing_paths = []

for image_path in tqdm(manifest["image_path"].astype(str), desc="Hashing Okra images"):
    path = Path(image_path)
    if not path.is_file():
        hashes.append(None)
        missing_paths.append(image_path)
        continue
    hashes.append(sha256_file(path))

manifest["sha256"] = hashes

valid = manifest.dropna(subset=["sha256"]).copy()
duplicate_rows = []

for sha, group in valid.groupby("sha256"):
    if len(group) < 2:
        continue

    duplicate_rows.append({
        "sha256": sha,
        "copies": len(group),
        "labels": ", ".join(sorted(group["label"].astype(str).unique())),
        "conflicting_labels": group["label"].nunique() > 1,
        "image_paths": " | ".join(group["image_path"].astype(str)),
    })

duplicates = pd.DataFrame(duplicate_rows)

hash_manifest_path = EXPERIMENT_DIR / "okra_manifest_with_sha256.csv"
duplicate_report_path = EXPERIMENT_DIR / "okra_exact_duplicate_report.csv"

manifest.to_csv(hash_manifest_path, index=False)
duplicates.to_csv(duplicate_report_path, index=False)

print("Images hashed:", len(valid))
print("Missing image paths:", len(missing_paths))
print("Exact duplicate groups:", len(duplicates))

if not duplicates.empty:
    print("Groups with conflicting labels:", int(duplicates["conflicting_labels"].sum()))
    print("Groups with consistent labels:", int((~duplicates["conflicting_labels"]).sum()))
    print("\nDuplicate groups:")
    display(duplicates.sort_values(
        ["conflicting_labels", "copies"],
        ascending=[False, False],
    ).head(30))
else:
    print("No exact duplicate groups found.")

print("\nHashed manifest:", hash_manifest_path)
print("Duplicate report:", duplicate_report_path)

Hashing Okra images:   0%|          | 0/4362 [00:00<?, ?it/s]

Images hashed: 4362
Missing image paths: 0
Exact duplicate groups: 36
Groups with conflicting labels: 0
Groups with consistent labels: 36

Duplicate groups:


,sha256,copies,labels,conflicting_labels,image_paths
32,edd00da96df3a8c15c0fca0d57b8dac37a1a4ebbcbac4b...,4,Healthy_Leaf,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...
0,063f8f8b785785fdddc52ef57ab970549065a535e191a3...,3,Healthy_Leaf,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...
3,156bd46a59fb5799412036d38aab2485aef22643469690...,3,Healthy_Leaf,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...
5,2a41b82ed1f2bc88da299bbd0a0d82147ef08adce16bdb...,3,Healthy_Leaf,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...
10,415ec50eaba173298cda5b195d874f400d16a57e4509d9...,3,Healthy_Leaf,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...
14,68cdc012f50e1a58bee4ef544f04b9bb74e2ebe195e0b1...,3,Healthy_Leaf,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...
15,6e441ea64647007bc3fa28beb0a2a3e9cb0c8486855014...,3,Healthy_Leaf,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...
16,6f3ecfbacaa979800937b6c1bfd5018ab24ea0635727da...,3,Healthy_Leaf,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...
17,737c345d74fb308292dc947b6c56c04a0a9b293035a323...,3,Healthy_Leaf,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...
23,97bbc3686a3a24627e73f355d2b73c0bea22f81eb3475c...,3,Healthy_Leaf,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...



Hashed manifest: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_manifest_with_sha256.csv
Duplicate report: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_exact_duplicate_report.csv


In [3]:
import pandas as pd
from pathlib import Path

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
HASHED_MANIFEST = EXPERIMENT_DIR / "okra_manifest_with_sha256.csv"
DUPLICATE_REPORT = EXPERIMENT_DIR / "okra_exact_duplicate_report.csv"

df = pd.read_csv(HASHED_MANIFEST)
duplicates = pd.read_csv(DUPLICATE_REPORT)

# Exact duplicates across different labels require review before cleanup.
if not duplicates.empty:
    conflicts = duplicates[duplicates["conflicting_labels"].astype(str).str.lower() == "true"]
    if not conflicts.empty:
        raise ValueError(
            "Found exact-duplicate groups with conflicting labels. "
            "Review them before cleanup."
        )

# Keep the first occurrence of each file hash; all duplicate copies are
# recorded in a manifest, while source image files remain untouched.
df = df.reset_index(drop=True)
duplicate_mask = df.duplicated(subset="sha256", keep="first")

removed = df.loc[duplicate_mask].copy()
kept = df.loc[~duplicate_mask].copy()

# Record which retained path represents each hash.
kept_path_by_hash = (
    kept.drop_duplicates("sha256")
    .set_index("sha256")["image_path"]
    .to_dict()
)
removed["kept_representative_path"] = removed["sha256"].map(kept_path_by_hash)
removed["set_aside_reason"] = "exact_duplicate_copy"

clean_path = EXPERIMENT_DIR / "okra_clean_manifest.csv"
removed_path = EXPERIMENT_DIR / "okra_exact_duplicates_set_aside.csv"

kept.to_csv(clean_path, index=False)
removed.to_csv(removed_path, index=False)

print("Images before exact-duplicate cleanup:", len(df))
print("Redundant exact copies set aside:", len(removed))
print("Images remaining:", len(kept))

print("\nRemaining images per class:")
display(kept["label"].value_counts().rename("images").to_frame())

print("\nSaved cleaned manifest:", clean_path)
print("Saved set-aside log:", removed_path)

Images before exact-duplicate cleanup: 4362
Redundant exact copies set aside: 50
Images remaining: 4312

Remaining images per class:


,images
label,
Yellow_Vein_Mosaic_Virus,1182
Healthy_Leaf,1101
Insects_Eaten_Leaf,743
Leaf_Curl,550
Leaf_Spot,479
Fruit_Rot,257



Saved cleaned manifest: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_clean_manifest.csv
Saved set-aside log: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_exact_duplicates_set_aside.csv


In [4]:
from pathlib import Path

OKRA_DIR = Path("/content/drive/MyDrive/final_year_project/okra")
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

if not OKRA_DIR.exists():
    raise FileNotFoundError(f"Okra folder not found: {OKRA_DIR}")

print("Okra folder contents:")
for item in sorted(OKRA_DIR.iterdir()):
    print(f"- {item.name}{'/' if item.is_dir() else ''}")

print("\nFolders that contain Healthy or Curl images:")
for folder in OKRA_DIR.rglob("*"):
    if not folder.is_dir():
        continue
    if "model_experiment" in folder.parts:
        continue

    name = folder.name.casefold()
    if "healthy" not in name and "curl" not in name:
        continue

    image_files = [
        path for path in folder.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    ]
    if image_files:
        print(f"{folder} — {len(image_files)} images")

Okra folder contents:
- Raw_Image_Dataset.zip
- healthy/
- okra_model_experiment/

Folders that contain Healthy or Curl images:
/content/drive/MyDrive/final_year_project/okra/healthy — 97 images
/content/drive/MyDrive/final_year_project/okra/healthy/Class 3 - Healthy — 78 images
/content/drive/MyDrive/final_year_project/okra/healthy/Class 4 - Leaf curly virus — 19 images


In [5]:
import pandas as pd
from pathlib import Path
from PIL import Image, ImageOps
from tqdm.auto import tqdm

OKRA_DIR = Path("/content/drive/MyDrive/final_year_project/okra")
EXPERIMENT_DIR = OKRA_DIR / "okra_model_experiment"

ORIGINAL_MANIFEST = EXPERIMENT_DIR / "okra_initial_manifest.csv"
ADDITIONS_ROOT = OKRA_DIR / "healthy"
COMBINED_MANIFEST = EXPERIMENT_DIR / "okra_combined_initial_manifest.csv"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

if not ORIGINAL_MANIFEST.exists():
    raise FileNotFoundError(f"Original manifest not found: {ORIGINAL_MANIFEST}")

class_folder_map = {
    "Class 3 - Healthy": "Healthy_Leaf",
    "Class 4 - Leaf curly virus": "Leaf_Curl",
}

original_df = pd.read_csv(ORIGINAL_MANIFEST)
addition_rows = []

for folder_name, label in class_folder_map.items():
    folder = ADDITIONS_ROOT / folder_name
    if not folder.is_dir():
        raise FileNotFoundError(f"Expected additional class folder not found: {folder}")

    image_files = sorted(
        path for path in folder.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )

    print(f"{label} additions found: {len(image_files)}")

    for image_path in tqdm(image_files, desc=f"Checking {label} additions"):
        try:
            with Image.open(image_path) as source:
                image = ImageOps.exif_transpose(source)
                width, height = image.size
                image.verify()

            addition_rows.append({
                "image_path": str(image_path),
                "label": label,
                "width": width,
                "height": height,
                "format": image_path.suffix.lower(),
                "dataset_source": "additional_drive_images",
            })
        except Exception as error:
            print(f"Unreadable image skipped: {image_path} ({error})")

original_df["dataset_source"] = "original_dataset"
additions_df = pd.DataFrame(addition_rows)

combined_df = pd.concat(
    [original_df, additions_df],
    ignore_index=True,
    sort=False,
)

# Guard against duplicate manifest paths.
if combined_df["image_path"].astype(str).duplicated().any():
    repeated = combined_df.loc[
        combined_df["image_path"].astype(str).duplicated(keep=False),
        "image_path",
    ]
    raise ValueError(f"Repeated manifest paths found:\n{repeated.tolist()[:10]}")

combined_df.to_csv(COMBINED_MANIFEST, index=False)

print("\nOriginal images:", len(original_df))
print("Additional readable images:", len(additions_df))
print("Combined images:", len(combined_df))
print("\nCombined class counts:")
display(combined_df["label"].value_counts().rename("images").to_frame())
print("\nCombined manifest saved to:", COMBINED_MANIFEST)

Healthy_Leaf additions found: 78


Checking Healthy_Leaf additions:   0%|          | 0/78 [00:00<?, ?it/s]

Leaf_Curl additions found: 19


Checking Leaf_Curl additions:   0%|          | 0/19 [00:00<?, ?it/s]


Original images: 4362
Additional readable images: 97
Combined images: 4459

Combined class counts:


,images
label,
Healthy_Leaf,1220
Yellow_Vein_Mosaic_Virus,1185
Insects_Eaten_Leaf,744
Leaf_Curl,573
Leaf_Spot,480
Fruit_Rot,257



Combined manifest saved to: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_combined_initial_manifest.csv


In [6]:
import hashlib
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
COMBINED_MANIFEST = EXPERIMENT_DIR / "okra_combined_initial_manifest.csv"

df = pd.read_csv(COMBINED_MANIFEST)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

hashes = []
missing_paths = []

for image_path in tqdm(df["image_path"].astype(str), desc="Hashing combined Okra images"):
    path = Path(image_path)
    if not path.is_file():
        hashes.append(None)
        missing_paths.append(image_path)
    else:
        hashes.append(sha256_file(path))

df["sha256"] = hashes
valid = df.dropna(subset=["sha256"]).copy()

duplicate_rows = []
for sha, group in valid.groupby("sha256"):
    if len(group) > 1:
        duplicate_rows.append({
            "sha256": sha,
            "copies": len(group),
            "labels": ", ".join(sorted(group["label"].astype(str).unique())),
            "conflicting_labels": group["label"].nunique() > 1,
            "image_paths": " | ".join(group["image_path"].astype(str)),
        })

duplicates = pd.DataFrame(duplicate_rows)

HASHED_PATH = EXPERIMENT_DIR / "okra_combined_manifest_with_sha256.csv"
DUPLICATE_REPORT = EXPERIMENT_DIR / "okra_combined_exact_duplicate_report.csv"

df.to_csv(HASHED_PATH, index=False)
duplicates.to_csv(DUPLICATE_REPORT, index=False)

print("Images in combined manifest:", len(df))
print("Images hashed:", len(valid))
print("Missing paths:", len(missing_paths))
print("Exact duplicate groups:", len(duplicates))

if not duplicates.empty:
    conflicts = duplicates["conflicting_labels"].astype(str).str.lower().eq("true")
    print("Groups with conflicting labels:", int(conflicts.sum()))
    print("Groups with consistent labels:", int((~conflicts).sum()))
    print("\nDuplicate groups:")
    display(duplicates.sort_values(
        ["conflicting_labels", "copies"],
        ascending=[False, False],
    ).head(40))
else:
    print("No exact duplicate groups found.")

print("\nHashed manifest saved:", HASHED_PATH)
print("Duplicate report saved:", DUPLICATE_REPORT)

Hashing combined Okra images:   0%|          | 0/4459 [00:00<?, ?it/s]

Images in combined manifest: 4459
Images hashed: 4459
Missing paths: 0
Exact duplicate groups: 65
Groups with conflicting labels: 0
Groups with consistent labels: 65

Duplicate groups:


,sha256,copies,labels,conflicting_labels,image_paths
59,edd00da96df3a8c15c0fca0d57b8dac37a1a4ebbcbac4b...,8,Healthy_Leaf,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...
0,063f8f8b785785fdddc52ef57ab970549065a535e191a3...,6,Healthy_Leaf,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...
6,156bd46a59fb5799412036d38aab2485aef22643469690...,6,Healthy_Leaf,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...
9,2a41b82ed1f2bc88da299bbd0a0d82147ef08adce16bdb...,6,Healthy_Leaf,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...
14,415ec50eaba173298cda5b195d874f400d16a57e4509d9...,6,Healthy_Leaf,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...
23,68cdc012f50e1a58bee4ef544f04b9bb74e2ebe195e0b1...,6,Healthy_Leaf,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...
24,6e441ea64647007bc3fa28beb0a2a3e9cb0c8486855014...,6,Healthy_Leaf,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...
25,6f3ecfbacaa979800937b6c1bfd5018ab24ea0635727da...,6,Healthy_Leaf,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...
26,737c345d74fb308292dc947b6c56c04a0a9b293035a323...,6,Healthy_Leaf,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...
41,97bbc3686a3a24627e73f355d2b73c0bea22f81eb3475c...,6,Healthy_Leaf,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...



Hashed manifest saved: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_combined_manifest_with_sha256.csv
Duplicate report saved: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_combined_exact_duplicate_report.csv


In [7]:
import pandas as pd
from pathlib import Path

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
HASHED_MANIFEST = EXPERIMENT_DIR / "okra_combined_manifest_with_sha256.csv"
DUPLICATE_REPORT = EXPERIMENT_DIR / "okra_combined_exact_duplicate_report.csv"

df = pd.read_csv(HASHED_MANIFEST).reset_index(drop=True)
duplicate_report = pd.read_csv(DUPLICATE_REPORT)

if df["sha256"].isna().any():
    raise ValueError("Some images have no SHA-256 hash. Resolve missing files before cleanup.")

if not duplicate_report.empty:
    conflict_mask = (
        duplicate_report["conflicting_labels"]
        .astype(str)
        .str.lower()
        .eq("true")
    )
    if conflict_mask.any():
        display(duplicate_report[conflict_mask])
        raise ValueError(
            "Conflicting-label duplicate groups need manual review before cleanup."
        )

duplicate_mask = df.duplicated(subset="sha256", keep="first")
kept = df.loc[~duplicate_mask].copy()
set_aside = df.loc[duplicate_mask].copy()

representatives = (
    kept.drop_duplicates("sha256")
    .set_index("sha256")["image_path"]
    .to_dict()
)
set_aside["kept_representative_path"] = set_aside["sha256"].map(representatives)
set_aside["set_aside_reason"] = "exact_duplicate_copy"

CLEAN_PATH = EXPERIMENT_DIR / "okra_clean_manifest.csv"
REMOVAL_LOG = EXPERIMENT_DIR / "okra_exact_duplicates_set_aside.csv"

kept.to_csv(CLEAN_PATH, index=False)
set_aside.to_csv(REMOVAL_LOG, index=False)

print("Images before exact-duplicate cleanup:", len(df))
print("Redundant exact copies set aside:", len(set_aside))
print("Images remaining:", len(kept))

print("\nRemaining images per class:")
display(kept["label"].value_counts().rename("images").to_frame())

print("\nClean manifest:", CLEAN_PATH)
print("Set-aside log:", REMOVAL_LOG)

Images before exact-duplicate cleanup: 4459
Redundant exact copies set aside: 147
Images remaining: 4312

Remaining images per class:


,images
label,
Yellow_Vein_Mosaic_Virus,1182
Healthy_Leaf,1101
Insects_Eaten_Leaf,743
Leaf_Curl,550
Leaf_Spot,479
Fruit_Rot,257



Clean manifest: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_clean_manifest.csv
Set-aside log: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_exact_duplicates_set_aside.csv


In [9]:
import pandas as pd
from pathlib import Path

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)

HASHED_PATH = EXPERIMENT_DIR / "okra_combined_manifest_with_sha256.csv"
CLEAN_PATH = EXPERIMENT_DIR / "okra_clean_manifest.csv"
SET_ASIDE_PATH = EXPERIMENT_DIR / "okra_exact_duplicates_set_aside.csv"

hashed = pd.read_csv(HASHED_PATH)
clean = pd.read_csv(CLEAN_PATH)
set_aside = pd.read_csv(SET_ASIDE_PATH)

original = hashed[hashed["dataset_source"] == "original_dataset"].copy()
additions = hashed[hashed["dataset_source"] == "additional_drive_images"].copy()

original_hashes = set(original["sha256"].dropna())
addition_hash_counts = additions["sha256"].value_counts()

# Determine whether each added file's image content already exists in the original dataset.
addition_audit = additions[
    ["image_path", "label", "sha256"]
].copy()
addition_audit["hash_exists_in_original"] = addition_audit["sha256"].isin(original_hashes)
addition_audit["duplicate_copies_within_additions"] = (
    addition_audit["sha256"].map(addition_hash_counts) > 1
)

clean_paths = set(clean["image_path"].astype(str))
addition_audit["retained_in_clean_manifest"] = (
    addition_audit["image_path"].astype(str).isin(clean_paths)
)

# Attach the representative path chosen by cleanup, if this added file was set aside.
representative_by_path = (
    set_aside.set_index("image_path")["kept_representative_path"].to_dict()
)
addition_audit["kept_representative_path"] = (
    addition_audit["image_path"].astype(str).map(representative_by_path)
)

audit_path = EXPERIMENT_DIR / "okra_additional_images_duplicate_audit.csv"
addition_audit.to_csv(audit_path, index=False)

print("Original dataset rows:", len(original))
print("Added image rows:", len(additions))
print("Unique added image hashes:", additions["sha256"].nunique())
print("Added hashes already present in original dataset:",
      additions.loc[additions["sha256"].isin(original_hashes), "sha256"].nunique())
print("Added images retained after exact cleanup:",
      int(addition_audit["retained_in_clean_manifest"].sum()))
print("Added images set aside:",
      int((~addition_audit["retained_in_clean_manifest"]).sum()))
print("Added images with content present in original dataset:",
      int(addition_audit["hash_exists_in_original"].sum()))

print("\nAudit counts by added class:")
display(
    addition_audit.groupby("label").agg(
        added_images=("image_path", "size"),
        unique_hashes=("sha256", "nunique"),
        already_in_original=("hash_exists_in_original", "sum"),
        retained=("retained_in_clean_manifest", "sum"),
    )
)

print("\nExamples of added files and their retained representatives:")
display(addition_audit.head(20))

print("\nDetailed audit saved to:", audit_path)

Original dataset rows: 4362
Added image rows: 97
Unique added image hashes: 56
Added hashes already present in original dataset: 56
Added images retained after exact cleanup: 0
Added images set aside: 97
Added images with content present in original dataset: 97

Audit counts by added class:


,added_images,unique_hashes,already_in_original,retained
label,,,,
Healthy_Leaf,78,37,78,0
Leaf_Curl,19,19,19,0



Examples of added files and their retained representatives:


,image_path,label,sha256,hash_exists_in_original,duplicate_copies_within_additions,retained_in_clean_manifest,kept_representative_path
4362,/content/drive/MyDrive/final_year_project/okra...,Healthy_Leaf,415ec50eaba173298cda5b195d874f400d16a57e4509d9...,True,True,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...
4363,/content/drive/MyDrive/final_year_project/okra...,Healthy_Leaf,415ec50eaba173298cda5b195d874f400d16a57e4509d9...,True,True,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...
4364,/content/drive/MyDrive/final_year_project/okra...,Healthy_Leaf,f524ff70398a4905038c883f10a9da20e89fb18d688bed...,True,True,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...
4365,/content/drive/MyDrive/final_year_project/okra...,Healthy_Leaf,0c6ef3acfa172542b97535d355b52df0a5512e28e1547d...,True,False,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...
4366,/content/drive/MyDrive/final_year_project/okra...,Healthy_Leaf,97bbc3686a3a24627e73f355d2b73c0bea22f81eb3475c...,True,True,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...
4367,/content/drive/MyDrive/final_year_project/okra...,Healthy_Leaf,2f7f56abedbe9c76527ff88ef314485f19691417e45b2a...,True,True,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...
4368,/content/drive/MyDrive/final_year_project/okra...,Healthy_Leaf,6f3ecfbacaa979800937b6c1bfd5018ab24ea0635727da...,True,True,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...
4369,/content/drive/MyDrive/final_year_project/okra...,Healthy_Leaf,2a41b82ed1f2bc88da299bbd0a0d82147ef08adce16bdb...,True,True,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...
4370,/content/drive/MyDrive/final_year_project/okra...,Healthy_Leaf,5456d7b835d03d59520103fe275dccc0dd5e5fc3bd1b8a...,True,True,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...
4371,/content/drive/MyDrive/final_year_project/okra...,Healthy_Leaf,6e441ea64647007bc3fa28beb0a2a3e9cb0c8486855014...,True,True,False,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...



Detailed audit saved to: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_additional_images_duplicate_audit.csv


In [10]:
!pip -q install ImageHash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 20.7 MB/s eta 0:00:00


In [11]:
import numpy as np
import pandas as pd
import imagehash
from pathlib import Path
from PIL import Image, ImageOps
from tqdm.auto import tqdm

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
CLEAN_MANIFEST = EXPERIMENT_DIR / "okra_clean_manifest.csv"

df = pd.read_csv(CLEAN_MANIFEST).reset_index(drop=True)

def rotation_tolerant_phash(image_path):
    with Image.open(image_path) as source:
        image = ImageOps.exif_transpose(source).convert("RGB")

    hashes = [
        int(str(imagehash.phash(image.rotate(angle, expand=True))))
        for angle in (0, 90, 180, 270)
    ]
    return min(hashes)

hash_values = []
for path in tqdm(df["image_path"].astype(str), desc="Hashing Okra images"):
    hash_values.append(rotation_tolerant_phash(path))

df["phash_rotation_tolerant"] = [f"{value:016x}" for value in hash_values]
hash_manifest_path = EXPERIMENT_DIR / "okra_perceptual_hashes.csv"
df.to_csv(hash_manifest_path, index=False)

# Compare image pairs by 64-bit perceptual-hash Hamming distance.
hash_array = np.asarray(hash_values, dtype=np.uint64)
lookup = np.array([bin(value).count("1") for value in range(256)], dtype=np.uint8)

MAX_DISTANCE = 12
candidate_rows = []

for i in tqdm(range(len(hash_array) - 1), desc="Comparing Okra pairs"):
    xor_values = np.bitwise_xor(hash_array[i + 1:], hash_array[i])
    byte_values = xor_values.view(np.uint8).reshape(-1, 8)
    distances = lookup[byte_values].sum(axis=1)

    for offset in np.flatnonzero(distances <= MAX_DISTANCE):
        j = i + 1 + int(offset)
        candidate_rows.append({
            "image_path_1": df.at[i, "image_path"],
            "label_1": df.at[i, "label"],
            "image_path_2": df.at[j, "image_path"],
            "label_2": df.at[j, "label"],
            "hamming_distance": int(distances[offset]),
            "different_labels": df.at[i, "label"] != df.at[j, "label"],
        })

candidates = pd.DataFrame(candidate_rows)
if not candidates.empty:
    candidates = candidates.sort_values(
        ["hamming_distance", "different_labels"],
        ascending=[True, False],
    ).reset_index(drop=True)

candidate_path = EXPERIMENT_DIR / "okra_near_duplicate_candidates.csv"
candidates.to_csv(candidate_path, index=False)

print("Unique images checked:", len(df))
print(f"Candidate pairs at distance <= {MAX_DISTANCE}:", len(candidates))
if not candidates.empty:
    print("Pairs with different labels:", int(candidates["different_labels"].sum()))
    print("\nCandidates by Hamming distance:")
    display(candidates["hamming_distance"].value_counts().sort_index().rename("pairs").to_frame())
    print("\nClosest pairs for review:")
    display(candidates.head(30))

print("\nPerceptual hashes saved:", hash_manifest_path)
print("Candidate pairs saved:", candidate_path)

Hashing Okra images:   0%|          | 0/4312 [00:00<?, ?it/s]

ValueError: invalid literal for int() with base 10: 'efb087d89e422f28'

In [12]:
import numpy as np
import pandas as pd
import imagehash
from pathlib import Path
from PIL import Image, ImageOps
from tqdm.auto import tqdm

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
CLEAN_MANIFEST = EXPERIMENT_DIR / "okra_clean_manifest.csv"

df = pd.read_csv(CLEAN_MANIFEST).reset_index(drop=True)

def rotation_tolerant_phash(image_path):
    with Image.open(image_path) as source:
        image = ImageOps.exif_transpose(source).convert("RGB")

    # ImageHash prints its 64-bit value as hexadecimal.
    return min(
        int(str(imagehash.phash(image.rotate(angle, expand=True))), 16)
        for angle in (0, 90, 180, 270)
    )

hash_values = [
    rotation_tolerant_phash(path)
    for path in tqdm(df["image_path"].astype(str), desc="Hashing Okra images")
]

df["phash_rotation_tolerant"] = [f"{value:016x}" for value in hash_values]
hash_manifest_path = EXPERIMENT_DIR / "okra_perceptual_hashes.csv"
df.to_csv(hash_manifest_path, index=False)

hash_array = np.asarray(hash_values, dtype=np.uint64)
lookup = np.array([bin(value).count("1") for value in range(256)], dtype=np.uint8)

MAX_DISTANCE = 12
candidate_rows = []

for i in tqdm(range(len(hash_array) - 1), desc="Comparing Okra pairs"):
    xor_values = np.bitwise_xor(hash_array[i + 1:], hash_array[i])
    byte_values = xor_values.view(np.uint8).reshape(-1, 8)
    distances = lookup[byte_values].sum(axis=1)

    for offset in np.flatnonzero(distances <= MAX_DISTANCE):
        j = i + 1 + int(offset)
        candidate_rows.append({
            "image_path_1": df.at[i, "image_path"],
            "label_1": df.at[i, "label"],
            "image_path_2": df.at[j, "image_path"],
            "label_2": df.at[j, "label"],
            "hamming_distance": int(distances[offset]),
            "different_labels": df.at[i, "label"] != df.at[j, "label"],
        })

candidates = pd.DataFrame(candidate_rows)
if not candidates.empty:
    candidates = candidates.sort_values(
        ["hamming_distance", "different_labels"],
        ascending=[True, False],
    ).reset_index(drop=True)

candidate_path = EXPERIMENT_DIR / "okra_near_duplicate_candidates.csv"
candidates.to_csv(candidate_path, index=False)

print("Unique images checked:", len(df))
print(f"Candidate pairs at distance <= {MAX_DISTANCE}:", len(candidates))

if not candidates.empty:
    print("Pairs with different labels:", int(candidates["different_labels"].sum()))
    print("\nCandidates by Hamming distance:")
    display(candidates["hamming_distance"].value_counts().sort_index().rename("pairs").to_frame())
    print("\nClosest pairs for review:")
    display(candidates.head(30))
else:
    print("No near-duplicate candidates found.")

print("\nPerceptual hashes saved:", hash_manifest_path)
print("Candidate pairs saved:", candidate_path)

Hashing Okra images:   0%|          | 0/4312 [00:00<?, ?it/s]

Comparing Okra pairs:   0%|          | 0/4311 [00:00<?, ?it/s]

Unique images checked: 4312
Candidate pairs at distance <= 12: 3413
Pairs with different labels: 29

Candidates by Hamming distance:


,pairs
hamming_distance,
0,2763
2,498
4,25
6,22
8,22
10,21
12,62



Closest pairs for review:


,image_path_1,label_1,image_path_2,label_2,hamming_distance,different_labels
0,/content/okra_raw/Raw_Image_Dataset/Fruit_Rot/...,Fruit_Rot,/content/okra_raw/Raw_Image_Dataset/Fruit_Rot/...,Fruit_Rot,0,False
1,/content/okra_raw/Raw_Image_Dataset/Fruit_Rot/...,Fruit_Rot,/content/okra_raw/Raw_Image_Dataset/Fruit_Rot/...,Fruit_Rot,0,False
2,/content/okra_raw/Raw_Image_Dataset/Fruit_Rot/...,Fruit_Rot,/content/okra_raw/Raw_Image_Dataset/Fruit_Rot/...,Fruit_Rot,0,False
3,/content/okra_raw/Raw_Image_Dataset/Fruit_Rot/...,Fruit_Rot,/content/okra_raw/Raw_Image_Dataset/Fruit_Rot/...,Fruit_Rot,0,False
4,/content/okra_raw/Raw_Image_Dataset/Fruit_Rot/...,Fruit_Rot,/content/okra_raw/Raw_Image_Dataset/Fruit_Rot/...,Fruit_Rot,0,False
5,/content/okra_raw/Raw_Image_Dataset/Fruit_Rot/...,Fruit_Rot,/content/okra_raw/Raw_Image_Dataset/Fruit_Rot/...,Fruit_Rot,0,False
6,/content/okra_raw/Raw_Image_Dataset/Fruit_Rot/...,Fruit_Rot,/content/okra_raw/Raw_Image_Dataset/Fruit_Rot/...,Fruit_Rot,0,False
7,/content/okra_raw/Raw_Image_Dataset/Fruit_Rot/...,Fruit_Rot,/content/okra_raw/Raw_Image_Dataset/Fruit_Rot/...,Fruit_Rot,0,False
8,/content/okra_raw/Raw_Image_Dataset/Fruit_Rot/...,Fruit_Rot,/content/okra_raw/Raw_Image_Dataset/Fruit_Rot/...,Fruit_Rot,0,False
9,/content/okra_raw/Raw_Image_Dataset/Fruit_Rot/...,Fruit_Rot,/content/okra_raw/Raw_Image_Dataset/Fruit_Rot/...,Fruit_Rot,0,False



Perceptual hashes saved: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_perceptual_hashes.csv
Candidate pairs saved: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_near_duplicate_candidates.csv


In [13]:
import math
import pandas as pd
from pathlib import Path
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
CANDIDATES_PATH = EXPERIMENT_DIR / "okra_near_duplicate_candidates.csv"

candidates = pd.read_csv(CANDIDATES_PATH)
if candidates.empty:
    raise ValueError("The near-duplicate candidate file is empty.")

# Review the 30 closest candidates first. The candidate file is sorted by
# Hamming distance, with cross-label pairs prioritized at equal distances.
START_ROW = 0
PAIR_COUNT = 30
review = candidates.iloc[START_ROW:START_ROW + PAIR_COUNT].copy().reset_index(drop=True)
review.insert(0, "review_pair", range(START_ROW, START_ROW + len(review)))

review_csv = EXPERIMENT_DIR / f"okra_near_duplicate_review_{START_ROW}_{START_ROW + len(review) - 1}.csv"
sheet_path = EXPERIMENT_DIR / f"okra_near_duplicate_review_{START_ROW}_{START_ROW + len(review) - 1}.png"
review.to_csv(review_csv, index=False)

thumb_w, thumb_h = 190, 170
label_h = 42
pairs_per_row = 2
rows = math.ceil(len(review) / pairs_per_row)

sheet = Image.new(
    "RGB",
    (pairs_per_row * 2 * thumb_w, rows * (thumb_h + label_h)),
    "white",
)
draw = ImageDraw.Draw(sheet)

for pair_index, row in review.iterrows():
    sheet_row = pair_index // pairs_per_row
    pair_column = pair_index % pairs_per_row

    for side in (1, 2):
        image_path = Path(str(row[f"image_path_{side}"]))
        x = pair_column * 2 * thumb_w + (side - 1) * thumb_w
        y = sheet_row * (thumb_h + label_h)

        if not image_path.is_file():
            draw.text((x + 6, y + 8), "MISSING IMAGE", fill="red")
            continue

        try:
            with Image.open(image_path) as source:
                image = ImageOps.exif_transpose(source).convert("RGB")
                image.thumbnail((thumb_w - 8, thumb_h - 8))
                tile = Image.new("RGB", (thumb_w, thumb_h), "white")
                tile.paste(
                    image,
                    ((thumb_w - image.width) // 2, (thumb_h - image.height) // 2),
                )
                sheet.paste(tile, (x, y))

            label = str(row[f"label_{side}"])
            draw.text((x + 5, y + thumb_h + 3), f"Pair {row['review_pair']} | {label}", fill="black")
        except Exception as error:
            draw.text((x + 5, y + 8), f"Could not open: {error}", fill="red")

    distance_text = f"distance={row['hamming_distance']}"
    cross_label_text = " | CROSS-LABEL" if bool(row["different_labels"]) else ""
    draw.text(
        (pair_column * 2 * thumb_w + 5, sheet_row * (thumb_h + label_h) + thumb_h + 21),
        distance_text + cross_label_text,
        fill="blue" if cross_label_text else "gray",
    )

sheet.save(sheet_path)

print(f"Review pairs: {len(review)} (rows {START_ROW}–{START_ROW + len(review) - 1})")
print("Pair list saved to:", review_csv)
print("Contact sheet saved to:", sheet_path)
display(sheet)

Output hidden; open in https://colab.research.google.com to view.

In [14]:
import pandas as pd
from pathlib import Path

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
CLEAN_MANIFEST = EXPERIMENT_DIR / "okra_clean_manifest.csv"

df = pd.read_csv(CLEAN_MANIFEST)

DROP_CLASS = "Fruit_Rot"
if DROP_CLASS not in set(df["label"].astype(str)):
    raise ValueError(f"Class {DROP_CLASS!r} was not found in the clean manifest.")

fruit_rot_set_aside = df[df["label"].astype(str) == DROP_CLASS].copy()
okra_df = df[df["label"].astype(str) != DROP_CLASS].copy().reset_index(drop=True)

fruit_rot_set_aside["set_aside_reason"] = "class_excluded_from_model"

OKRA_MANIFEST = EXPERIMENT_DIR / "okra_manifest_without_fruit_rot.csv"
SET_ASIDE_LOG = EXPERIMENT_DIR / "okra_fruit_rot_class_set_aside.csv"

okra_df.to_csv(OKRA_MANIFEST, index=False)
fruit_rot_set_aside.to_csv(SET_ASIDE_LOG, index=False)

print("Fruit_Rot images set aside:", len(fruit_rot_set_aside))
print("Images remaining:", len(okra_df))
print("\nClasses retained:")
display(okra_df["label"].value_counts().rename("images").to_frame())
print("\nWorking manifest saved to:", OKRA_MANIFEST)
print("Fruit_Rot set-aside log saved to:", SET_ASIDE_LOG)

Fruit_Rot images set aside: 257
Images remaining: 4055

Classes retained:


,images
label,
Yellow_Vein_Mosaic_Virus,1182
Healthy_Leaf,1101
Insects_Eaten_Leaf,743
Leaf_Curl,550
Leaf_Spot,479



Working manifest saved to: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_manifest_without_fruit_rot.csv
Fruit_Rot set-aside log saved to: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_fruit_rot_class_set_aside.csv


In [15]:
import numpy as np
import pandas as pd
import imagehash
from pathlib import Path
from PIL import Image, ImageOps
from tqdm.auto import tqdm

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
MANIFEST_PATH = EXPERIMENT_DIR / "okra_manifest_without_fruit_rot.csv"

df = pd.read_csv(MANIFEST_PATH).reset_index(drop=True)

def rotation_tolerant_phash(image_path):
    with Image.open(image_path) as source:
        image = ImageOps.exif_transpose(source).convert("RGB")

    return min(
        int(str(imagehash.phash(image.rotate(angle, expand=True))), 16)
        for angle in (0, 90, 180, 270)
    )

hash_values = [
    rotation_tolerant_phash(path)
    for path in tqdm(df["image_path"].astype(str), desc="Hashing five-class Okra images")
]

df["phash_rotation_tolerant"] = [f"{value:016x}" for value in hash_values]

HASH_MANIFEST = EXPERIMENT_DIR / "okra_five_class_perceptual_hashes.csv"
df.to_csv(HASH_MANIFEST, index=False)

hash_array = np.asarray(hash_values, dtype=np.uint64)
lookup = np.array([bin(value).count("1") for value in range(256)], dtype=np.uint8)

MAX_HAMMING_DISTANCE = 12
candidate_rows = []

for i in tqdm(range(len(hash_array) - 1), desc="Comparing five-class Okra pairs"):
    xor_values = np.bitwise_xor(hash_array[i + 1:], hash_array[i])
    byte_values = xor_values.view(np.uint8).reshape(-1, 8)
    distances = lookup[byte_values].sum(axis=1)

    for offset in np.flatnonzero(distances <= MAX_HAMMING_DISTANCE):
        j = i + 1 + int(offset)
        candidate_rows.append({
            "image_path_1": df.at[i, "image_path"],
            "label_1": df.at[i, "label"],
            "image_path_2": df.at[j, "image_path"],
            "label_2": df.at[j, "label"],
            "hamming_distance": int(distances[offset]),
            "different_labels": df.at[i, "label"] != df.at[j, "label"],
        })

candidates = pd.DataFrame(candidate_rows)
if not candidates.empty:
    candidates = candidates.sort_values(
        ["hamming_distance", "different_labels"],
        ascending=[True, False],
    ).reset_index(drop=True)

CANDIDATE_PATH = EXPERIMENT_DIR / "okra_five_class_near_duplicate_candidates.csv"
candidates.to_csv(CANDIDATE_PATH, index=False)

print("Images checked:", len(df))
print(f"Candidate pairs at distance <= {MAX_HAMMING_DISTANCE}:", len(candidates))

if not candidates.empty:
    print("Pairs with different labels:", int(candidates["different_labels"].sum()))
    print("\nCandidate count by distance:")
    display(
        candidates["hamming_distance"]
        .value_counts()
        .sort_index()
        .rename("pairs")
        .to_frame()
    )

    print("\nCross-label candidate count by class pair:")
    cross_label = candidates[candidates["different_labels"]].copy()
    if not cross_label.empty:
        cross_label["class_pair"] = cross_label.apply(
            lambda row: " | ".join(sorted([row["label_1"], row["label_2"]])),
            axis=1,
        )
        display(cross_label["class_pair"].value_counts().rename("pairs").to_frame())
    else:
        print("No cross-label candidates found.")

    print("\nClosest candidate pairs:")
    display(candidates.head(30))
else:
    print("No near-duplicate candidates found.")

print("\nFive-class hashes saved to:", HASH_MANIFEST)
print("Five-class candidate pairs saved to:", CANDIDATE_PATH)

Hashing five-class Okra images:   0%|          | 0/4055 [00:00<?, ?it/s]

Comparing five-class Okra pairs:   0%|          | 0/4054 [00:00<?, ?it/s]

Images checked: 4055
Candidate pairs at distance <= 12: 3115
Pairs with different labels: 29

Candidate count by distance:


,pairs
hamming_distance,
0,2576
2,444
4,7
6,10
8,7
10,16
12,55



Cross-label candidate count by class pair:


,pairs
class_pair,
Healthy_Leaf | Yellow_Vein_Mosaic_Virus,12
Insects_Eaten_Leaf | Leaf_Spot,6
Leaf_Spot | Yellow_Vein_Mosaic_Virus,6
Insects_Eaten_Leaf | Yellow_Vein_Mosaic_Virus,5



Closest candidate pairs:


,image_path_1,label_1,image_path_2,label_2,hamming_distance,different_labels
0,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False
1,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False
2,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False
3,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False
4,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False
5,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False
6,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False
7,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False
8,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False
9,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False



Five-class hashes saved to: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_five_class_perceptual_hashes.csv
Five-class candidate pairs saved to: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_five_class_near_duplicate_candidates.csv


In [16]:
import numpy as np
import pandas as pd
import imagehash
from pathlib import Path
from PIL import Image, ImageOps
from tqdm.auto import tqdm

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
MANIFEST_PATH = EXPERIMENT_DIR / "okra_manifest_without_fruit_rot.csv"

df = pd.read_csv(MANIFEST_PATH).reset_index(drop=True)

def rotation_tolerant_phash_256(image_path):
    with Image.open(image_path) as source:
        image = ImageOps.exif_transpose(source).convert("RGB")

    # 16 x 16 hash gives 256 bits, reducing collisions from the earlier 64-bit hash.
    return min(
        int(str(imagehash.phash(image.rotate(angle, expand=True), hash_size=16)), 16)
        for angle in (0, 90, 180, 270)
    )

hash_values = [
    rotation_tolerant_phash_256(path)
    for path in tqdm(df["image_path"].astype(str), desc="Computing 256-bit Okra hashes")
]

# Store each 256-bit integer as 32 bytes for vectorized Hamming-distance comparisons.
hash_matrix = np.stack([
    np.frombuffer(value.to_bytes(32, byteorder="big"), dtype=np.uint8)
    for value in hash_values
])

hash_manifest = df.copy()
hash_manifest["phash_256_rotation_tolerant"] = [
    f"{value:064x}" for value in hash_values
]
HASH_MANIFEST_PATH = EXPERIMENT_DIR / "okra_five_class_perceptual_hashes_256bit.csv"
hash_manifest.to_csv(HASH_MANIFEST_PATH, index=False)

byte_popcount = np.array([int(i).bit_count() for i in range(256)], dtype=np.uint8)
MAX_DISTANCE = 32  # Out of 256 bits; candidates still require visual confirmation.

candidate_rows = []

for i in tqdm(range(len(hash_matrix) - 1), desc="Comparing 256-bit Okra hashes"):
    xor_bytes = np.bitwise_xor(hash_matrix[i + 1:], hash_matrix[i])
    distances = byte_popcount[xor_bytes].sum(axis=1)

    for offset in np.flatnonzero(distances <= MAX_DISTANCE):
        j = i + 1 + int(offset)
        candidate_rows.append({
            "image_path_1": df.at[i, "image_path"],
            "label_1": df.at[i, "label"],
            "image_path_2": df.at[j, "image_path"],
            "label_2": df.at[j, "label"],
            "hamming_distance": int(distances[offset]),
            "different_labels": df.at[i, "label"] != df.at[j, "label"],
        })

candidates = pd.DataFrame(candidate_rows)
if not candidates.empty:
    candidates = candidates.sort_values(
        ["hamming_distance", "different_labels"],
        ascending=[True, False],
    ).reset_index(drop=True)

CANDIDATE_PATH = EXPERIMENT_DIR / "okra_five_class_near_duplicate_candidates_256bit.csv"
candidates.to_csv(CANDIDATE_PATH, index=False)

print("Images checked:", len(df))
print("Hash size: 256 bits")
print(f"Candidate pairs at distance <= {MAX_DISTANCE}:", len(candidates))

if not candidates.empty:
    print("Pairs with different labels:", int(candidates["different_labels"].sum()))
    print("\nCandidate count by distance:")
    display(
        candidates["hamming_distance"]
        .value_counts()
        .sort_index()
        .rename("pairs")
        .to_frame()
    )
    print("\nClosest candidate pairs:")
    display(candidates.head(30))
else:
    print("No candidates found at this threshold.")

print("\n256-bit hash manifest:", HASH_MANIFEST_PATH)
print("256-bit candidate report:", CANDIDATE_PATH)

Computing 256-bit Okra hashes:   0%|          | 0/4055 [00:00<?, ?it/s]

Comparing 256-bit Okra hashes:   0%|          | 0/4054 [00:00<?, ?it/s]

Images checked: 4055
Hash size: 256 bits
Candidate pairs at distance <= 32: 3046
Pairs with different labels: 0

Candidate count by distance:


,pairs
hamming_distance,
0,1517
2,1198
4,214
6,51
8,34
10,15
12,2
14,2
16,1



Closest candidate pairs:


,image_path_1,label_1,image_path_2,label_2,hamming_distance,different_labels
0,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False
1,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False
2,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False
3,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False
4,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False
5,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False
6,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False
7,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False
8,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False
9,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False



256-bit hash manifest: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_five_class_perceptual_hashes_256bit.csv
256-bit candidate report: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_five_class_near_duplicate_candidates_256bit.csv


In [17]:
import math
import pandas as pd
from pathlib import Path
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
CANDIDATES_PATH = (
    EXPERIMENT_DIR / "okra_five_class_near_duplicate_candidates_256bit.csv"
)

candidates = pd.read_csv(CANDIDATES_PATH)
if candidates.empty:
    raise ValueError("The candidate-pairs file is empty.")

# Review the closest pairs first. The report is sorted by distance.
start = 0
count = 30
review = candidates.iloc[start:start + count].copy().reset_index(drop=True)
review.insert(0, "review_pair", range(start, start + len(review)))

review_csv = EXPERIMENT_DIR / "okra_near_duplicate_review_256bit_0_29.csv"
sheet_path = EXPERIMENT_DIR / "okra_near_duplicate_review_256bit_0_29.png"
review.to_csv(review_csv, index=False)

thumb_w, thumb_h = 220, 190
label_h = 54
pairs_per_row = 2
rows = math.ceil(len(review) / pairs_per_row)

sheet = Image.new(
    "RGB",
    (pairs_per_row * 2 * thumb_w, rows * (thumb_h + label_h)),
    "white",
)
draw = ImageDraw.Draw(sheet)

for pair_index, row in review.iterrows():
    sheet_row = pair_index // pairs_per_row
    pair_column = pair_index % pairs_per_row
    row_y = sheet_row * (thumb_h + label_h)

    for side in (1, 2):
        image_path = Path(str(row[f"image_path_{side}"]))
        x = pair_column * 2 * thumb_w + (side - 1) * thumb_w

        if not image_path.is_file():
            draw.text((x + 6, row_y + 8), "IMAGE FILE MISSING", fill="red")
            continue

        try:
            with Image.open(image_path) as source:
                image = ImageOps.exif_transpose(source).convert("RGB")
                image.thumbnail((thumb_w - 10, thumb_h - 10))
                tile = Image.new("RGB", (thumb_w, thumb_h), "white")
                tile.paste(
                    image,
                    ((thumb_w - image.width) // 2, (thumb_h - image.height) // 2),
                )
                sheet.paste(tile, (x, row_y))

            filename = image_path.name
            label = str(row[f"label_{side}"])
            draw.text(
                (x + 5, row_y + thumb_h + 2),
                f"{filename[:27]} | {label}",
                fill="black",
            )
        except Exception as error:
            draw.text((x + 5, row_y + 8), f"Could not open: {error}", fill="red")

    note = f"Pair {row['review_pair']} | distance={row['hamming_distance']}"
    draw.text(
        (pair_column * 2 * thumb_w + 5, row_y + thumb_h + 23),
        note,
        fill="blue",
    )

sheet.save(sheet_path)

print(f"Showing candidate pairs {start}–{start + len(review) - 1}")
print("Pair list:", review_csv)
print("Contact sheet:", sheet_path)
display(sheet)

Output hidden; open in https://colab.research.google.com to view.

In [18]:
import math
import pandas as pd
from pathlib import Path
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
CANDIDATES_PATH = (
    EXPERIMENT_DIR / "okra_five_class_near_duplicate_candidates_256bit.csv"
)

candidates = pd.read_csv(CANDIDATES_PATH)
pair_count = 30
pairs = candidates.head(pair_count).copy().reset_index(drop=True)

if pairs.empty:
    raise ValueError("No candidate pairs are available to review.")

# Union-find groups paths that are connected through one or more candidate pairs.
parent = {}

def find(path):
    parent.setdefault(path, path)
    if parent[path] != path:
        parent[path] = find(parent[path])
    return parent[path]

def union(path_a, path_b):
    root_a = find(path_a)
    root_b = find(path_b)
    if root_a != root_b:
        parent[root_b] = root_a

for _, row in pairs.iterrows():
    union(str(row["image_path_1"]), str(row["image_path_2"]))

components = {}
for image_path in list(parent):
    root = find(image_path)
    components.setdefault(root, []).append(image_path)

# Map each image path to its class label.
path_to_label = {}
for _, row in pairs.iterrows():
    path_to_label[str(row["image_path_1"])] = str(row["label_1"])
    path_to_label[str(row["image_path_2"])] = str(row["label_2"])

components = sorted(components.values(), key=lambda group: (-len(group), group[0]))

component_rows = []
for component_id, paths in enumerate(components, start=1):
    labels = sorted({path_to_label[path] for path in paths})
    for path in sorted(paths):
        component_rows.append({
            "review_group": component_id,
            "group_size": len(paths),
            "labels": ", ".join(labels),
            "label": path_to_label[path],
            "image_path": path,
        })

group_df = pd.DataFrame(component_rows)
groups_path = EXPERIMENT_DIR / "okra_near_duplicate_review_groups_0_29.csv"
group_df.to_csv(groups_path, index=False)

# Create one grouped contact sheet. Each connected group gets its own section.
thumb_w, thumb_h = 210, 185
label_h = 38
columns = 4
section_heights = [
    34 + math.ceil(len(paths) / columns) * (thumb_h + label_h)
    for paths in components
]
sheet = Image.new(
    "RGB",
    (columns * thumb_w, sum(section_heights)),
    "white",
)
draw = ImageDraw.Draw(sheet)

y_offset = 0
for component_id, paths in enumerate(components, start=1):
    paths = sorted(paths)
    labels = sorted({path_to_label[path] for path in paths})
    draw.text(
        (8, y_offset + 8),
        f"Review group {component_id} | {len(paths)} unique images | {', '.join(labels)}",
        fill="blue",
    )
    content_y = y_offset + 34

    for image_index, image_path in enumerate(paths):
        row_index = image_index // columns
        column_index = image_index % columns
        x = column_index * thumb_w
        y = content_y + row_index * (thumb_h + label_h)

        try:
            with Image.open(image_path) as source:
                image = ImageOps.exif_transpose(source).convert("RGB")
                image.thumbnail((thumb_w - 10, thumb_h - 10))
                tile = Image.new("RGB", (thumb_w, thumb_h), "white")
                tile.paste(
                    image,
                    ((thumb_w - image.width) // 2, (thumb_h - image.height) // 2),
                )
                sheet.paste(tile, (x, y))

            draw.text(
                (x + 4, y + thumb_h + 2),
                f"{Path(image_path).name[:25]} | {path_to_label[image_path]}",
                fill="black",
            )
        except Exception as error:
            draw.text((x + 4, y + 8), f"Could not open image: {error}", fill="red")

    y_offset += section_heights[component_id - 1]

sheet_path = EXPERIMENT_DIR / "okra_near_duplicate_review_groups_0_29.png"
sheet.save(sheet_path)

print(f"Candidate pairs grouped: {len(pairs)}")
print(f"Connected review groups: {len(components)}")
print("Group membership CSV:", groups_path)
print("Grouped contact sheet:", sheet_path)
display(sheet)

Output hidden; open in https://colab.research.google.com to view.

In [19]:
import pandas as pd
from pathlib import Path

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)

MANIFEST_PATH = EXPERIMENT_DIR / "okra_manifest_without_fruit_rot.csv"
GROUPS_PATH = EXPERIMENT_DIR / "okra_near_duplicate_review_groups_0_29.csv"

df = pd.read_csv(MANIFEST_PATH)
groups = pd.read_csv(GROUPS_PATH)

if groups.empty:
    raise ValueError(f"No reviewed groups found in {GROUPS_PATH}")

# Check every image in the review groups exists in the current manifest.
manifest_paths = set(df["image_path"].astype(str))
group_paths = set(groups["image_path"].astype(str))
missing = group_paths - manifest_paths
if missing:
    raise ValueError(
        f"{len(missing)} review-group images are missing from the manifest. "
        f"Examples: {list(missing)[:5]}"
    )

# Check that each candidate group stays within one class.
label_counts = groups.groupby("review_group")["label"].nunique()
if (label_counts > 1).any():
    bad_groups = label_counts[label_counts > 1].index.tolist()
    raise ValueError(
        f"Cross-label groups need label review before cleanup: {bad_groups}"
    )

# Keep one deterministic representative in each reviewed group.
# All images outside these groups remain untouched.
keep_paths = set()
set_aside_rows = []

for group_id, group in groups.groupby("review_group", sort=True):
    paths = sorted(group["image_path"].astype(str).unique())
    representative = paths[0]
    keep_paths.add(representative)

    for image_path in paths[1:]:
        row = df.loc[df["image_path"].astype(str) == image_path].iloc[0].to_dict()
        row["review_group"] = int(group_id)
        row["kept_representative_path"] = representative
        row["set_aside_reason"] = "visually_reviewed_near_duplicate_group"
        set_aside_rows.append(row)

set_aside_paths = {
    row["image_path"] for row in set_aside_rows
}

cleaned = df[~df["image_path"].astype(str).isin(set_aside_paths)].copy()
set_aside_df = pd.DataFrame(set_aside_rows)

CLEANED_PATH = EXPERIMENT_DIR / "okra_manifest_after_near_duplicate_batch_0_29.csv"
SET_ASIDE_PATH = EXPERIMENT_DIR / "okra_near_duplicates_set_aside_batch_0_29.csv"

cleaned.to_csv(CLEANED_PATH, index=False)
set_aside_df.to_csv(SET_ASIDE_PATH, index=False)

print("Images before this reviewed batch:", len(df))
print("Images set aside from reviewed groups:", len(set_aside_df))
print("Images remaining:", len(cleaned))
print("\nRemaining images per class:")
display(cleaned["label"].value_counts().rename("images").to_frame())

print("\nUpdated working manifest:", CLEANED_PATH)
print("Set-aside log:", SET_ASIDE_PATH)

Images before this reviewed batch: 4055
Images set aside from reviewed groups: 29
Images remaining: 4026

Remaining images per class:


,images
label,
Yellow_Vein_Mosaic_Virus,1182
Healthy_Leaf,1101
Insects_Eaten_Leaf,743
Leaf_Curl,550
Leaf_Spot,450



Updated working manifest: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_manifest_after_near_duplicate_batch_0_29.csv
Set-aside log: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_near_duplicates_set_aside_batch_0_29.csv


In [20]:
import numpy as np
import pandas as pd
import imagehash
from pathlib import Path
from PIL import Image, ImageOps
from tqdm.auto import tqdm

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
MANIFEST_PATH = (
    EXPERIMENT_DIR / "okra_manifest_after_near_duplicate_batch_0_29.csv"
)
PREVIOUS_REVIEW_PATH = (
    EXPERIMENT_DIR / "okra_near_duplicate_review_256bit_0_29.csv"
)

df = pd.read_csv(MANIFEST_PATH).reset_index(drop=True)

def dihedral_phash_256(image_path):
    """Rotation- and mirror-tolerant 256-bit pHash."""
    with Image.open(image_path) as source:
        image = ImageOps.exif_transpose(source).convert("RGB")

    variants = []
    for base in (image, ImageOps.mirror(image)):
        for angle in (0, 90, 180, 270):
            transformed = base.rotate(angle, expand=True)
            variants.append(
                int(str(imagehash.phash(transformed, hash_size=16)), 16)
            )
    return min(variants)

hash_values = [
    dihedral_phash_256(path)
    for path in tqdm(df["image_path"].astype(str), desc="Hashing remaining Okra images")
]

hash_matrix = np.stack([
    np.frombuffer(value.to_bytes(32, byteorder="big"), dtype=np.uint8)
    for value in hash_values
])

hash_manifest = df.copy()
hash_manifest["phash_256_dihedral"] = [f"{value:064x}" for value in hash_values]
HASH_PATH = EXPERIMENT_DIR / "okra_remaining_perceptual_hashes_256bit_dihedral.csv"
hash_manifest.to_csv(HASH_PATH, index=False)

# Skip exact path pairs already reviewed in the first 30-pair batch.
previously_reviewed = set()
if PREVIOUS_REVIEW_PATH.exists():
    previous = pd.read_csv(PREVIOUS_REVIEW_PATH)
    for _, row in previous.iterrows():
        previously_reviewed.add(
            frozenset((str(row["image_path_1"]), str(row["image_path_2"])))
        )

byte_popcount = np.array([int(i).bit_count() for i in range(256)], dtype=np.uint8)
MAX_DISTANCE = 48  # Wider search for transformed or lightly altered copies.

candidate_rows = []

for i in tqdm(range(len(hash_matrix) - 1), desc="Comparing remaining Okra images"):
    xor_bytes = np.bitwise_xor(hash_matrix[i + 1:], hash_matrix[i])
    distances = byte_popcount[xor_bytes].sum(axis=1)

    for offset in np.flatnonzero(distances <= MAX_DISTANCE):
        j = i + 1 + int(offset)
        path1 = str(df.at[i, "image_path"])
        path2 = str(df.at[j, "image_path"])

        if frozenset((path1, path2)) in previously_reviewed:
            continue

        candidate_rows.append({
            "image_path_1": path1,
            "label_1": df.at[i, "label"],
            "image_path_2": path2,
            "label_2": df.at[j, "label"],
            "hamming_distance": int(distances[offset]),
            "different_labels": df.at[i, "label"] != df.at[j, "label"],
        })

candidates = pd.DataFrame(candidate_rows)
if not candidates.empty:
    candidates = candidates.sort_values(
        ["hamming_distance", "different_labels"],
        ascending=[True, False],
    ).reset_index(drop=True)

CANDIDATE_PATH = (
    EXPERIMENT_DIR / "okra_remaining_augmented_near_duplicate_candidates_48.csv"
)
candidates.to_csv(CANDIDATE_PATH, index=False)

print("Remaining images checked:", len(df))
print("Transform checks: 4 rotations × original/mirrored")
print(f"Candidate pairs at distance <= {MAX_DISTANCE}:", len(candidates))

if not candidates.empty:
    print("Pairs with different labels:", int(candidates["different_labels"].sum()))
    print("\nCandidate count by distance:")
    display(
        candidates["hamming_distance"]
        .value_counts()
        .sort_index()
        .rename("pairs")
        .to_frame()
    )
    print("\nClosest candidates:")
    display(candidates.head(30))
else:
    print("No candidates found at this threshold.")

print("\nHash manifest:", HASH_PATH)
print("Candidate report:", CANDIDATE_PATH)


Hashing remaining Okra images:   0%|          | 0/4026 [00:00<?, ?it/s]

Comparing remaining Okra images:   0%|          | 0/4025 [00:00<?, ?it/s]

Remaining images checked: 4026
Transform checks: 4 rotations × original/mirrored
Candidate pairs at distance <= 48: 2904
Pairs with different labels: 0

Candidate count by distance:


,pairs
hamming_distance,
0,1406
2,1163
4,242
6,31
8,33
10,11
12,3
14,2
26,2



Closest candidates:


,image_path_1,label_1,image_path_2,label_2,hamming_distance,different_labels
0,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False
1,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False
2,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False
3,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False
4,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False
5,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False
6,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False
7,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False
8,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False
9,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,/content/okra_raw/Raw_Image_Dataset/Leaf_Spot/...,Leaf_Spot,0,False



Hash manifest: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_remaining_perceptual_hashes_256bit_dihedral.csv
Candidate report: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_remaining_augmented_near_duplicate_candidates_48.csv


In [21]:
import math
from pathlib import Path
from collections import defaultdict

import pandas as pd
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
PAIR_FILE = EXPERIMENT_DIR / "okra_remaining_augmented_near_duplicate_candidates_48.csv"
OUTPUT_DIR = EXPERIMENT_DIR / "okra_remaining_duplicate_review"

if not PAIR_FILE.exists():
    raise FileNotFoundError(f"Candidate-pairs file not found: {PAIR_FILE}")

pairs = pd.read_csv(PAIR_FILE)
required = {"image_path_1", "image_path_2", "label_1", "label_2", "hamming_distance"}
missing = required - set(pairs.columns)
if missing:
    raise ValueError(f"Candidate file is missing columns: {sorted(missing)}")

# Build connected components so an image that appears in multiple pairs
# is shown only once within its review group.
parent = {}

def find(x):
    parent.setdefault(x, x)
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

for row in pairs.itertuples(index=False):
    union(str(row.image_path_1), str(row.image_path_2))

components = defaultdict(set)
for path in list(parent):
    components[find(path)].add(path)

# Keep groups with at least two images; sort by their closest pair distance.
group_rows = []
for paths in components.values():
    if len(paths) < 2:
        continue
    subset = pairs[
        pairs["image_path_1"].astype(str).isin(paths)
        & pairs["image_path_2"].astype(str).isin(paths)
    ]
    group_rows.append({
        "paths": sorted(paths),
        "min_distance": int(subset["hamming_distance"].min()),
        "pair_count": len(subset),
        "labels": sorted(set(subset["label_1"].astype(str)) |
                         set(subset["label_2"].astype(str))),
    })

group_rows.sort(key=lambda x: (x["min_distance"], -len(x["paths"])))
print("Candidate pairs:", len(pairs))
print("Connected groups:", len(group_rows))
print("Groups with multiple labels:", sum(len(g["labels"]) > 1 for g in group_rows))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Make manageable contact sheets for the 30 closest groups.
MAX_GROUPS = 30
THUMB_W, THUMB_H = 220, 220
LABEL_H = 48
COLS = 4

for group_number, group in enumerate(group_rows[:MAX_GROUPS]):
    paths = group["paths"]
    rows = math.ceil(len(paths) / COLS)
    sheet = Image.new(
        "RGB",
        (COLS * THUMB_W, rows * (THUMB_H + LABEL_H)),
        "white",
    )
    draw = ImageDraw.Draw(sheet)

    for i, path in enumerate(paths):
        x = (i % COLS) * THUMB_W
        y = (i // COLS) * (THUMB_H + LABEL_H)
        try:
            with Image.open(path) as im:
                im = ImageOps.exif_transpose(im).convert("RGB")
                im.thumbnail((THUMB_W - 10, THUMB_H - 10))
                sheet.paste(im, (x + (THUMB_W - im.width) // 2,
                                 y + (THUMB_H - im.height) // 2))
            label = Path(path).parent.name
            filename = Path(path).name
            draw.text((x + 4, y + THUMB_H), f"{label} | {filename}"[:34], fill="black")
        except Exception as exc:
            draw.text((x + 4, y + 10), f"Could not open: {Path(path).name}", fill="red")

    sheet_path = OUTPUT_DIR / f"group_{group_number:03d}.jpg"
    sheet.save(sheet_path, quality=90)

    print(
        f"Group {group_number:03d}: {len(paths)} images; "
        f"closest hash distance={group['min_distance']}; "
        f"labels={group['labels']}; sheet={sheet_path}"
    )

    if group_number < 5:
        display(sheet)

print(f"\nSaved the first {min(MAX_GROUPS, len(group_rows))} group sheets to: {OUTPUT_DIR}")

Output hidden; open in https://colab.research.google.com to view.

In [22]:
import pandas as pd
from pathlib import Path

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
CURRENT_MANIFEST = EXPERIMENT_DIR / "okra_manifest_after_near_duplicate_batch_0_29.csv"
GROUP_DIR = EXPERIMENT_DIR / "okra_remaining_duplicate_review"
OUTPUT_MANIFEST = EXPERIMENT_DIR / "okra_manifest_after_remaining_30_groups.csv"
REMOVAL_LOG = EXPERIMENT_DIR / "okra_remaining_30_groups_set_aside.csv"

if not CURRENT_MANIFEST.exists():
    raise FileNotFoundError(f"Current manifest not found: {CURRENT_MANIFEST}")
if not GROUP_DIR.exists():
    raise FileNotFoundError(f"Review sheets folder not found: {GROUP_DIR}")

df = pd.read_csv(CURRENT_MANIFEST)
if not {"image_path", "label"}.issubset(df.columns):
    raise ValueError("Manifest must contain image_path and label columns.")

# Rebuild the same connected groups from the candidate-pairs file.
PAIR_FILE = EXPERIMENT_DIR / "okra_remaining_augmented_near_duplicate_candidates_48.csv"
pairs = pd.read_csv(PAIR_FILE)

parent = {}

def find(x):
    parent.setdefault(x, x)
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

for row in pairs.itertuples(index=False):
    union(str(row.image_path_1), str(row.image_path_2))

components = {}
for path in list(parent):
    components.setdefault(find(path), set()).add(path)

groups = [sorted(paths) for paths in components.values() if len(paths) > 1]
pairs_per_group = []
for paths in groups:
    subset = pairs[
        pairs["image_path_1"].astype(str).isin(paths)
        & pairs["image_path_2"].astype(str).isin(paths)
    ]
    min_distance = int(subset["hamming_distance"].min())
    pairs_per_group.append((min_distance, paths))

groups = [paths for _, paths in sorted(pairs_per_group, key=lambda x: (x[0], -len(x[1])))]

# These are the 30 group sheets created by the review cell.
confirmed_group_count = min(30, len(groups))
confirmed_groups = groups[:confirmed_group_count]

manifest_paths = set(df["image_path"].astype(str))
to_remove = set()
removal_rows = []

for group_number, paths in enumerate(confirmed_groups):
    missing = set(paths) - manifest_paths
    if missing:
        raise ValueError(
            f"Group {group_number:03d} contains {len(missing)} path(s) "
            "not found in the current manifest. No files were changed."
        )

    # Keep the first path in sorted order; set aside the rest.
    kept_path = paths[0]
    kept_label = df.loc[df["image_path"].astype(str) == kept_path, "label"].iloc[0]

    for removed_path in paths[1:]:
        removed_label = df.loc[
            df["image_path"].astype(str) == removed_path, "label"
        ].iloc[0]
        if removed_label != kept_label:
            raise ValueError(
                f"Conflicting labels in group {group_number}: "
                f"{kept_label} vs {removed_label}. No files were changed."
            )
        to_remove.add(removed_path)
        removal_rows.append({
            "group_number": group_number,
            "kept_image_path": kept_path,
            "set_aside_image_path": removed_path,
            "label": removed_label,
        })

updated = df[~df["image_path"].astype(str).isin(to_remove)].copy()

if len(updated) != len(df) - len(to_remove):
    raise RuntimeError("Row-count check failed; outputs were not saved.")

updated.to_csv(OUTPUT_MANIFEST, index=False)
pd.DataFrame(removal_rows).to_csv(REMOVAL_LOG, index=False)

print(f"Starting images: {len(df)}")
print(f"Confirmed groups processed: {confirmed_group_count}")
print(f"Images set aside: {len(to_remove)}")
print(f"Images remaining: {len(updated)}")
print("\nRemaining images by class:")
print(updated["label"].value_counts().sort_index().to_string())
print(f"\nUpdated manifest: {OUTPUT_MANIFEST}")
print(f"Set-aside log: {REMOVAL_LOG}")

Starting images: 4026
Confirmed groups processed: 30
Images set aside: 138
Images remaining: 3888

Remaining images by class:
label
Healthy_Leaf                1037
Insects_Eaten_Leaf           735
Leaf_Curl                    498
Leaf_Spot                    436
Yellow_Vein_Mosaic_Virus    1182

Updated manifest: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_manifest_after_remaining_30_groups.csv
Set-aside log: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_remaining_30_groups_set_aside.csv


In [23]:
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
MANIFEST_PATH = EXPERIMENT_DIR / "okra_manifest_after_remaining_30_groups.csv"
HASH_PATH = EXPERIMENT_DIR / "okra_remaining_perceptual_hashes_256bit_dihedral.csv"
OUTPUT_PATH = EXPERIMENT_DIR / "okra_higher_distance_candidates_50_80.csv"

MIN_DISTANCE = 50
MAX_DISTANCE = 80

manifest = pd.read_csv(MANIFEST_PATH)
hashes = pd.read_csv(HASH_PATH)

path_col = "image_path"
hash_col = next(
    (c for c in hashes.columns if "hash" in c.lower() and c != path_col),
    None,
)
if hash_col is None:
    raise ValueError(f"Could not identify the hash column. Columns: {hashes.columns.tolist()}")

if path_col not in hashes.columns:
    raise ValueError(f"Hash file has no {path_col!r} column.")

current_paths = set(manifest[path_col].astype(str))
hashes = hashes[hashes[path_col].astype(str).isin(current_paths)].copy()

if hashes[path_col].astype(str).duplicated().any():
    raise ValueError("Hash file contains duplicate image paths.")

# Convert hexadecimal hashes to integers and compare pairs.
path_to_label = dict(zip(manifest[path_col].astype(str), manifest["label"].astype(str)))
records = hashes[[path_col, hash_col]].dropna().to_records(index=False)
hash_items = [(str(path), int(str(value), 16)) for path, value in records]

candidates = []
for i in tqdm(range(len(hash_items)), desc="Comparing higher-distance Okra pairs"):
    path_a, hash_a = hash_items[i]
    for j in range(i + 1, len(hash_items)):
        path_b, hash_b = hash_items[j]
        distance = (hash_a ^ hash_b).bit_count()
        if MIN_DISTANCE <= distance <= MAX_DISTANCE:
            label_a = path_to_label[path_a]
            label_b = path_to_label[path_b]
            candidates.append({
                "image_path_1": path_a,
                "label_1": label_a,
                "image_path_2": path_b,
                "label_2": label_b,
                "hamming_distance": distance,
                "different_labels": label_a != label_b,
            })

candidate_df = pd.DataFrame(candidates)
candidate_df.to_csv(OUTPUT_PATH, index=False)

print("Images compared:", len(hash_items))
print(f"Candidate pairs at distances {MIN_DISTANCE}–{MAX_DISTANCE}:", len(candidate_df))
if len(candidate_df):
    print("Pairs with different labels:", int(candidate_df["different_labels"].sum()))
    print("\nCandidate count by distance:")
    print(candidate_df["hamming_distance"].value_counts().sort_index().to_string())
    display(candidate_df.sort_values("hamming_distance").head(30))
print("\nSaved candidates:", OUTPUT_PATH)

Comparing higher-distance Okra pairs:   0%|          | 0/3888 [00:00<?, ?it/s]

Images compared: 3888
Candidate pairs at distances 50–80: 19
Pairs with different labels: 0

Candidate count by distance:
hamming_distance
50    9
54    1
64    1
66    1
68    2
74    1
78    2
80    2


,image_path_1,label_1,image_path_2,label_2,hamming_distance,different_labels
7,/content/okra_raw/Raw_Image_Dataset/Yellow_Vei...,Yellow_Vein_Mosaic_Virus,/content/okra_raw/Raw_Image_Dataset/Yellow_Vei...,Yellow_Vein_Mosaic_Virus,50,False
6,/content/okra_raw/Raw_Image_Dataset/Yellow_Vei...,Yellow_Vein_Mosaic_Virus,/content/okra_raw/Raw_Image_Dataset/Yellow_Vei...,Yellow_Vein_Mosaic_Virus,50,False
5,/content/okra_raw/Raw_Image_Dataset/Yellow_Vei...,Yellow_Vein_Mosaic_Virus,/content/okra_raw/Raw_Image_Dataset/Yellow_Vei...,Yellow_Vein_Mosaic_Virus,50,False
4,/content/okra_raw/Raw_Image_Dataset/Yellow_Vei...,Yellow_Vein_Mosaic_Virus,/content/okra_raw/Raw_Image_Dataset/Yellow_Vei...,Yellow_Vein_Mosaic_Virus,50,False
15,/content/okra_raw/Raw_Image_Dataset/Yellow_Vei...,Yellow_Vein_Mosaic_Virus,/content/okra_raw/Raw_Image_Dataset/Yellow_Vei...,Yellow_Vein_Mosaic_Virus,50,False
10,/content/okra_raw/Raw_Image_Dataset/Yellow_Vei...,Yellow_Vein_Mosaic_Virus,/content/okra_raw/Raw_Image_Dataset/Yellow_Vei...,Yellow_Vein_Mosaic_Virus,50,False
9,/content/okra_raw/Raw_Image_Dataset/Yellow_Vei...,Yellow_Vein_Mosaic_Virus,/content/okra_raw/Raw_Image_Dataset/Yellow_Vei...,Yellow_Vein_Mosaic_Virus,50,False
8,/content/okra_raw/Raw_Image_Dataset/Yellow_Vei...,Yellow_Vein_Mosaic_Virus,/content/okra_raw/Raw_Image_Dataset/Yellow_Vei...,Yellow_Vein_Mosaic_Virus,50,False
17,/content/okra_raw/Raw_Image_Dataset/Yellow_Vei...,Yellow_Vein_Mosaic_Virus,/content/okra_raw/Raw_Image_Dataset/Yellow_Vei...,Yellow_Vein_Mosaic_Virus,50,False
0,/content/okra_raw/Raw_Image_Dataset/Leaf_Curl/...,Leaf_Curl,/content/okra_raw/Raw_Image_Dataset/Leaf_Curl/...,Leaf_Curl,54,False



Saved candidates: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_higher_distance_candidates_50_80.csv


In [24]:
from pathlib import Path
import math
import pandas as pd
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
PAIR_FILE = EXPERIMENT_DIR / "okra_higher_distance_candidates_50_80.csv"
OUTPUT_DIR = EXPERIMENT_DIR / "okra_higher_distance_review_50_80"

pairs = pd.read_csv(PAIR_FILE)
if pairs.empty:
    print("No candidate pairs to review.")
else:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    CELL_W, CELL_H = 220, 245
    PAIRS_PER_SHEET = 12
    PAIRS_PER_ROW = 2
    PAIR_ROW_H = CELL_H + 35

    def add_image(canvas, path, x, y, label):
        try:
            with Image.open(path) as im:
                im = ImageOps.exif_transpose(im).convert("RGB")
                im.thumbnail((CELL_W - 8, CELL_H - 45))
                canvas.paste(
                    im,
                    (x + (CELL_W - im.width) // 2, y + 4)
                )
            draw = ImageDraw.Draw(canvas)
            draw.text((x + 4, y + CELL_H - 38), label[:30], fill="black")
            draw.text((x + 4, y + CELL_H - 20), Path(path).name[:30], fill="black")
        except Exception as exc:
            ImageDraw.Draw(canvas).text(
                (x + 4, y + 8), f"Could not open image: {exc}", fill="red"
            )

    for start in range(0, len(pairs), PAIRS_PER_SHEET):
        batch = pairs.iloc[start:start + PAIRS_PER_SHEET]
        sheet = Image.new(
            "RGB",
            (CELL_W * 2, PAIR_ROW_H * len(batch)),
            "white",
        )
        draw = ImageDraw.Draw(sheet)

        for local_i, row in enumerate(batch.itertuples(index=False)):
            y = local_i * PAIR_ROW_H
            draw.text(
                (4, y + 2),
                f"Pair {start + local_i} | distance={row.hamming_distance}",
                fill="blue",
            )
            add_image(sheet, row.image_path_1, 0, y + 24, row.label_1)
            add_image(sheet, row.image_path_2, CELL_W, y + 24, row.label_2)

        sheet_path = OUTPUT_DIR / f"pairs_{start:03d}_{start + len(batch) - 1:03d}.jpg"
        sheet.save(sheet_path, quality=92)
        print(f"Saved: {sheet_path}")
        display(sheet)

Output hidden; open in https://colab.research.google.com to view.

In [25]:
from pathlib import Path
from collections import defaultdict
import pandas as pd
from tqdm.auto import tqdm

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
CURRENT_MANIFEST = EXPERIMENT_DIR / "okra_manifest_after_remaining_30_groups.csv"
CONFIRMED_PAIRS = EXPERIMENT_DIR / "okra_higher_distance_candidates_50_80.csv"
HASH_FILE = EXPERIMENT_DIR / "okra_remaining_perceptual_hashes_256bit_dihedral.csv"

UPDATED_MANIFEST = EXPERIMENT_DIR / "okra_manifest_after_confirmed_50_80_pairs.csv"
REMOVAL_LOG = EXPERIMENT_DIR / "okra_set_aside_confirmed_50_80_pairs.csv"
NEXT_CANDIDATES = EXPERIMENT_DIR / "okra_candidates_distance_2_50.csv"

MIN_DISTANCE, MAX_DISTANCE = 2, 50

for path in (CURRENT_MANIFEST, CONFIRMED_PAIRS, HASH_FILE):
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

manifest = pd.read_csv(CURRENT_MANIFEST)
pairs = pd.read_csv(CONFIRMED_PAIRS)
hash_df = pd.read_csv(HASH_FILE)

required_manifest = {"image_path", "label"}
required_pairs = {"image_path_1", "image_path_2", "label_1", "label_2"}
if not required_manifest.issubset(manifest.columns):
    raise ValueError(f"Manifest must include {sorted(required_manifest)}")
if not required_pairs.issubset(pairs.columns):
    raise ValueError(f"Pair file must include {sorted(required_pairs)}")
if len(pairs) != 19:
    raise ValueError(f"Expected the 19 reviewed pairs, found {len(pairs)}.")

manifest["image_path"] = manifest["image_path"].astype(str)
manifest_paths = set(manifest["image_path"])
labels = dict(zip(manifest["image_path"], manifest["label"].astype(str)))

# Merge overlapping pairs so each duplicate image is handled once.
parent = {}

def find(x):
    parent.setdefault(x, x)
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

for row in pairs.itertuples(index=False):
    a, b = str(row.image_path_1), str(row.image_path_2)
    if a not in manifest_paths or b not in manifest_paths:
        raise ValueError(f"Pair path is absent from the current manifest: {a} | {b}")
    if labels[a] != labels[b]:
        raise ValueError(f"Pair has conflicting labels: {labels[a]} vs {labels[b]}")
    union(a, b)

groups_by_root = defaultdict(set)
for path in parent:
    groups_by_root[find(path)].add(path)

groups = [sorted(paths) for paths in groups_by_root.values() if len(paths) > 1]
to_set_aside = set()
removal_rows = []

for group_id, group_paths in enumerate(groups):
    keep_path = group_paths[0]
    for remove_path in group_paths[1:]:
        to_set_aside.add(remove_path)
        removal_rows.append({
            "group_id": group_id,
            "kept_image_path": keep_path,
            "set_aside_image_path": remove_path,
            "label": labels[keep_path],
        })

updated = manifest[~manifest["image_path"].isin(to_set_aside)].copy()
updated.to_csv(UPDATED_MANIFEST, index=False)
pd.DataFrame(removal_rows).to_csv(REMOVAL_LOG, index=False)

print("Confirmed pairs:", len(pairs))
print("Duplicate groups after merging overlaps:", len(groups))
print("Images set aside:", len(to_set_aside))
print("Images remaining:", len(updated))
print("\nRemaining images by class:")
print(updated["label"].value_counts().sort_index().to_string())
print("\nUpdated manifest:", UPDATED_MANIFEST)
print("Set-aside log:", REMOVAL_LOG)

# Search only images in the updated manifest. The hash file stores one
# representative 256-bit dihedral hash per image.
hash_col = next(
    (c for c in hash_df.columns if "hash" in c.lower() and c != "image_path"),
    None,
)
if hash_col is None:
    raise ValueError(f"Could not find hash column. Hash file columns: {hash_df.columns.tolist()}")

hash_df["image_path"] = hash_df["image_path"].astype(str)
hash_df = hash_df[hash_df["image_path"].isin(set(updated["image_path"]))].copy()

if hash_df["image_path"].duplicated().any():
    raise ValueError("Hash file has repeated image paths.")
if hash_df[hash_col].isna().any():
    raise ValueError("Hash file contains missing hashes.")

items = [
    (str(path), int(str(value), 16))
    for path, value in hash_df[["image_path", hash_col]].itertuples(index=False, name=None)
]
updated_labels = dict(zip(updated["image_path"], updated["label"].astype(str)))

candidate_rows = []
for i in tqdm(range(len(items)), desc="Searching remaining Okra pairs"):
    path_a, hash_a = items[i]
    for j in range(i + 1, len(items)):
        path_b, hash_b = items[j]
        distance = (hash_a ^ hash_b).bit_count()
        if MIN_DISTANCE <= distance <= MAX_DISTANCE:
            label_a, label_b = updated_labels[path_a], updated_labels[path_b]
            candidate_rows.append({
                "image_path_1": path_a,
                "label_1": label_a,
                "image_path_2": path_b,
                "label_2": label_b,
                "hamming_distance": distance,
                "different_labels": label_a != label_b,
            })

candidate_df = pd.DataFrame(candidate_rows)
candidate_df.to_csv(NEXT_CANDIDATES, index=False)

print(f"\nRemaining images hashed: {len(items)}")
print(f"Candidate pairs at distances {MIN_DISTANCE}–{MAX_DISTANCE}: {len(candidate_df)}")
if not candidate_df.empty:
    print("Pairs with different labels:", int(candidate_df["different_labels"].sum()))
    print("\nCandidate count by distance:")
    print(candidate_df["hamming_distance"].value_counts().sort_index().to_string())
    display(candidate_df.sort_values("hamming_distance").head(30))
print("\nSaved next candidate list:", NEXT_CANDIDATES)

Confirmed pairs: 19
Duplicate groups after merging overlaps: 9
Images set aside: 17
Images remaining: 3871

Remaining images by class:
label
Healthy_Leaf                1034
Insects_Eaten_Leaf           735
Leaf_Curl                    497
Leaf_Spot                    436
Yellow_Vein_Mosaic_Virus    1169

Updated manifest: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_manifest_after_confirmed_50_80_pairs.csv
Set-aside log: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_set_aside_confirmed_50_80_pairs.csv


Searching remaining Okra pairs:   0%|          | 0/3871 [00:00<?, ?it/s]


Remaining images hashed: 3871
Candidate pairs at distances 2–50: 1279
Pairs with different labels: 0

Candidate count by distance:
hamming_distance
2     1024
4      197
6       22
8       25
10       7
12       1
34       3


,image_path_1,label_1,image_path_2,label_2,hamming_distance,different_labels
797,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...,Healthy_Leaf,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...,Healthy_Leaf,2,False
808,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...,Healthy_Leaf,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...,Healthy_Leaf,2,False
807,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...,Healthy_Leaf,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...,Healthy_Leaf,2,False
806,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...,Healthy_Leaf,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...,Healthy_Leaf,2,False
805,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...,Healthy_Leaf,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...,Healthy_Leaf,2,False
804,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...,Healthy_Leaf,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...,Healthy_Leaf,2,False
803,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...,Healthy_Leaf,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...,Healthy_Leaf,2,False
800,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...,Healthy_Leaf,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...,Healthy_Leaf,2,False
798,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...,Healthy_Leaf,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...,Healthy_Leaf,2,False
811,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...,Healthy_Leaf,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...,Healthy_Leaf,2,False



Saved next candidate list: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_candidates_distance_2_50.csv


In [26]:
from pathlib import Path
from collections import defaultdict
import math

import pandas as pd
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
PAIR_FILE = EXPERIMENT_DIR / "okra_candidates_distance_2_50.csv"
OUTPUT_DIR = EXPERIMENT_DIR / "okra_distance_2_50_group_review"

if not PAIR_FILE.exists():
    raise FileNotFoundError(f"Candidate file not found: {PAIR_FILE}")

pairs = pd.read_csv(PAIR_FILE)
required = {
    "image_path_1", "label_1", "image_path_2", "label_2", "hamming_distance"
}
missing = required - set(pairs.columns)
if missing:
    raise ValueError(f"Candidate file missing columns: {sorted(missing)}")

# Group overlapping candidate pairs into connected components.
parent = {}

def find(path):
    parent.setdefault(path, path)
    if parent[path] != path:
        parent[path] = find(parent[path])
    return parent[path]

def union(a, b):
    root_a, root_b = find(a), find(b)
    if root_a != root_b:
        parent[root_b] = root_a

for row in pairs.itertuples(index=False):
    union(str(row.image_path_1), str(row.image_path_2))

components = defaultdict(set)
for path in list(parent):
    components[find(path)].add(path)

groups = []
for paths in components.values():
    if len(paths) < 2:
        continue
    paths = sorted(paths)
    member_pairs = pairs[
        pairs["image_path_1"].astype(str).isin(paths)
        & pairs["image_path_2"].astype(str).isin(paths)
    ]
    groups.append({
        "paths": paths,
        "min_distance": int(member_pairs["hamming_distance"].min()),
        "pair_count": len(member_pairs),
        "labels": sorted(
            set(member_pairs["label_1"].astype(str))
            | set(member_pairs["label_2"].astype(str))
        ),
    })

groups.sort(key=lambda group: (group["min_distance"], -len(group["paths"])))

print("Candidate pairs:", len(pairs))
print("Connected groups:", len(groups))
print("Images across groups:", sum(len(group["paths"]) for group in groups))
print("Groups with multiple labels:", sum(len(group["labels"]) > 1 for group in groups))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GROUP_LIMIT = 30
COLS = 4
THUMB_W, THUMB_H, LABEL_H = 220, 200, 46

for group_id, group in enumerate(groups[:GROUP_LIMIT]):
    paths = group["paths"]
    row_count = math.ceil(len(paths) / COLS)
    sheet = Image.new(
        "RGB",
        (COLS * THUMB_W, row_count * (THUMB_H + LABEL_H) + 30),
        "white",
    )
    draw = ImageDraw.Draw(sheet)
    draw.text(
        (5, 5),
        f"Group {group_id} | min distance={group['min_distance']} | "
        f"pairs={group['pair_count']} | labels={group['labels']}",
        fill="blue",
    )

    for i, path in enumerate(paths):
        x = (i % COLS) * THUMB_W
        y = 30 + (i // COLS) * (THUMB_H + LABEL_H)
        try:
            with Image.open(path) as source:
                image = ImageOps.exif_transpose(source).convert("RGB")
                image.thumbnail((THUMB_W - 10, THUMB_H - 10))
                sheet.paste(
                    image,
                    (x + (THUMB_W - image.width) // 2, y + 4),
                )
            draw.text((x + 4, y + THUMB_H), Path(path).parent.name[:26], fill="black")
            draw.text((x + 4, y + THUMB_H + 17), Path(path).name[:28], fill="black")
        except Exception as exc:
            draw.text((x + 4, y + 8), f"Could not open: {exc}", fill="red")

    sheet_path = OUTPUT_DIR / f"group_{group_id:03d}.jpg"
    sheet.save(sheet_path, quality=90)
    print(f"Saved group {group_id}: {sheet_path}")

    if group_id < 5:
        display(sheet)

print(f"\nSaved review sheets for the first {min(GROUP_LIMIT, len(groups))} groups.")
print("Review folder:", OUTPUT_DIR)

Output hidden; open in https://colab.research.google.com to view.

In [27]:
from pathlib import Path
from collections import defaultdict
import pandas as pd

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
CURRENT_MANIFEST = EXPERIMENT_DIR / "okra_manifest_after_confirmed_50_80_pairs.csv"
PAIR_FILE = EXPERIMENT_DIR / "okra_candidates_distance_2_50.csv"

UPDATED_MANIFEST = EXPERIMENT_DIR / "okra_manifest_after_distance_2_50_groups.csv"
REMOVAL_LOG = EXPERIMENT_DIR / "okra_set_aside_distance_2_50_groups.csv"

if not CURRENT_MANIFEST.exists():
    raise FileNotFoundError(f"Current manifest not found: {CURRENT_MANIFEST}")
if not PAIR_FILE.exists():
    raise FileNotFoundError(f"Candidate-pairs file not found: {PAIR_FILE}")

manifest = pd.read_csv(CURRENT_MANIFEST)
pairs = pd.read_csv(PAIR_FILE)

if not {"image_path", "label"}.issubset(manifest.columns):
    raise ValueError("Manifest must contain image_path and label.")
required_pairs = {"image_path_1", "label_1", "image_path_2", "label_2", "hamming_distance"}
if not required_pairs.issubset(pairs.columns):
    raise ValueError(f"Candidate file must contain: {sorted(required_pairs)}")

manifest["image_path"] = manifest["image_path"].astype(str)
pairs["image_path_1"] = pairs["image_path_1"].astype(str)
pairs["image_path_2"] = pairs["image_path_2"].astype(str)
current_paths = set(manifest["image_path"])
path_labels = dict(zip(manifest["image_path"], manifest["label"].astype(str)))

# Confirm the candidate file contains only the requested distance range.
if not pairs.empty and not pairs["hamming_distance"].between(2, 50).all():
    raise ValueError("Candidate file contains pairs outside distances 2–50.")

# Merge overlapping pairs. Each connected component is treated as one
# confirmed duplicate group, retaining one deterministic representative.
parent = {}

def find(path):
    parent.setdefault(path, path)
    if parent[path] != path:
        parent[path] = find(parent[path])
    return parent[path]

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

for row in pairs.itertuples(index=False):
    a, b = row.image_path_1, row.image_path_2
    if a not in current_paths or b not in current_paths:
        raise ValueError(f"Candidate image is missing from current manifest: {a} | {b}")
    if path_labels[a] != path_labels[b]:
        raise ValueError(
            f"Conflicting labels in candidate pair: {path_labels[a]} vs {path_labels[b]}"
        )
    union(a, b)

components = defaultdict(set)
for path in parent:
    components[find(path)].add(path)

groups = [sorted(paths) for paths in components.values() if len(paths) > 1]
to_set_aside = set()
removal_rows = []

for group_id, group_paths in enumerate(groups):
    kept_path = group_paths[0]
    for removed_path in group_paths[1:]:
        to_set_aside.add(removed_path)
        removal_rows.append({
            "group_id": group_id,
            "kept_image_path": kept_path,
            "set_aside_image_path": removed_path,
            "label": path_labels[kept_path],
            "group_size": len(group_paths),
        })

updated = manifest[~manifest["image_path"].isin(to_set_aside)].copy()

if len(updated) != len(manifest) - len(to_set_aside):
    raise RuntimeError("Row-count validation failed; outputs were not written.")

updated.to_csv(UPDATED_MANIFEST, index=False)
pd.DataFrame(removal_rows).to_csv(REMOVAL_LOG, index=False)

print("Candidate pairs processed:", len(pairs))
print("Duplicate groups formed after merging overlaps:", len(groups))
print("Images set aside:", len(to_set_aside))
print("Images remaining:", len(updated))
print("\nRemaining images by class:")
print(updated["label"].value_counts().sort_index().to_string())
print("\nUpdated manifest:", UPDATED_MANIFEST)
print("Set-aside log:", REMOVAL_LOG)

Candidate pairs processed: 1279
Duplicate groups formed after merging overlaps: 643
Images set aside: 1072
Images remaining: 2799

Remaining images by class:
label
Healthy_Leaf                693
Insects_Eaten_Leaf          575
Leaf_Curl                   343
Leaf_Spot                   302
Yellow_Vein_Mosaic_Virus    886

Updated manifest: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_manifest_after_distance_2_50_groups.csv
Set-aside log: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_set_aside_distance_2_50_groups.csv


In [28]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image, ImageOps
from tqdm.auto import tqdm

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
MANIFEST_PATH = EXPERIMENT_DIR / "okra_manifest_after_distance_2_50_groups.csv"
REPORT_PATH = EXPERIMENT_DIR / "okra_image_clarity_audit.csv"

df = pd.read_csv(MANIFEST_PATH)
if "image_path" not in df.columns:
    raise ValueError("Manifest must contain an image_path column.")

rows = []
for path in tqdm(df["image_path"].astype(str), desc="Auditing remaining Okra images"):
    record = {"image_path": path, "readable": False, "width": None, "height": None,
              "mean_laplacian_variance": None, "error": None}
    try:
        with Image.open(path) as source:
            image = ImageOps.exif_transpose(source).convert("RGB")
            record["width"], record["height"] = image.size
            gray = np.asarray(image.convert("L"), dtype=np.float32)

            # Simple Laplacian variance as a blur screening signal.
            center = gray[1:-1, 1:-1]
            laplacian = (
                gray[:-2, 1:-1] + gray[2:, 1:-1]
                + gray[1:-1, :-2] + gray[1:-1, 2:]
                - 4 * center
            )
            record["mean_laplacian_variance"] = float(laplacian.var())
            record["readable"] = True
    except Exception as exc:
        record["error"] = str(exc)

    rows.append(record)

audit = pd.DataFrame(rows)
audit.to_csv(REPORT_PATH, index=False)

print("Images audited:", len(audit))
print("Readable:", int(audit["readable"].sum()))
print("Unreadable:", int((~audit["readable"]).sum()))
print("\nImage dimensions:")
print(audit.loc[audit["readable"], ["width", "height"]].value_counts().head(10).to_string())

print("\nBlur-screening score summary (lower can indicate blur; review visually):")
print(audit["mean_laplacian_variance"].describe().round(2).to_string())

print("\nLowest 20 blur-screening scores for visual inspection:")
display(
    audit[audit["readable"]]
    .nsmallest(20, "mean_laplacian_variance")[
        ["image_path", "width", "height", "mean_laplacian_variance"]
    ]
)

if (~audit["readable"]).any():
    print("\nUnreadable image paths:")
    display(audit.loc[~audit["readable"], ["image_path", "error"]])

print("\nAudit report saved to:", REPORT_PATH)

Auditing remaining Okra images:   0%|          | 0/2799 [00:00<?, ?it/s]

Images audited: 2799
Readable: 2799
Unreadable: 0

Image dimensions:
width  height
300    300       2746
6000   4000        42
2096   2122         1
3191   3432         1
2906   3276         1
3229   2836         1
3316   3047         1
3710   3325         1
3489   3693         1
3725   3510         1

Blur-screening score summary (lower can indicate blur; review visually):
count     2799.00
mean      4071.27
std       2935.66
min          4.52
25%       2261.36
50%       3318.45
75%       4956.75
max      28017.07

Lowest 20 blur-screening scores for visual inspection:


,image_path,width,height,mean_laplacian_variance
1062,/content/okra_raw/Raw_Image_Dataset/Leaf_Curl/...,4605,4000,4.518104
1000,/content/okra_raw/Raw_Image_Dataset/Leaf_Curl/...,4282,3404,5.107131
911,/content/okra_raw/Raw_Image_Dataset/Leaf_Curl/...,3316,3047,5.592314
1299,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...,6000,4000,6.435936
1032,/content/okra_raw/Raw_Image_Dataset/Leaf_Curl/...,3725,3510,6.585614
1143,/content/okra_raw/Raw_Image_Dataset/Leaf_Curl/...,4000,6000,7.735834
1559,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...,3489,3693,7.778025
1006,/content/okra_raw/Raw_Image_Dataset/Leaf_Curl/...,6000,4000,8.368011
1570,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...,6000,4000,8.684761
1783,/content/okra_raw/Raw_Image_Dataset/Healthy_Le...,6000,4000,8.957648



Audit report saved to: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_image_clarity_audit.csv


In [29]:
from pathlib import Path
import math
import pandas as pd
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
AUDIT_FILE = EXPERIMENT_DIR / "okra_image_clarity_audit.csv"
OUTPUT_FILE = EXPERIMENT_DIR / "okra_low_clarity_review.jpg"

audit = pd.read_csv(AUDIT_FILE)
audit = audit[audit["readable"] == True].dropna(
    subset=["mean_laplacian_variance"]
)
review = audit.nsmallest(40, "mean_laplacian_variance").reset_index(drop=True)

CELL_W, CELL_H = 240, 250
LABEL_H = 42
COLS = 4
ROWS = math.ceil(len(review) / COLS)

sheet = Image.new(
    "RGB",
    (COLS * CELL_W, ROWS * (CELL_H + LABEL_H)),
    "white",
)
draw = ImageDraw.Draw(sheet)

for i, row in review.iterrows():
    x = (i % COLS) * CELL_W
    y = (i // COLS) * (CELL_H + LABEL_H)

    try:
        with Image.open(row["image_path"]) as source:
            image = ImageOps.exif_transpose(source).convert("RGB")
            image.thumbnail((CELL_W - 10, CELL_H - 10))
            sheet.paste(
                image,
                (x + (CELL_W - image.width) // 2,
                 y + (CELL_H - image.height) // 2),
            )
        draw.text(
            (x + 4, y + CELL_H),
            f"score={row['mean_laplacian_variance']:.1f}",
            fill="blue",
        )
        draw.text(
            (x + 4, y + CELL_H + 18),
            Path(row["image_path"]).name[:32],
            fill="black",
        )
    except Exception as exc:
        draw.text((x + 4, y + 8), f"Could not open: {exc}", fill="red")

sheet.save(OUTPUT_FILE, quality=92)
print(f"Images shown: {len(review)}")
print(f"Contact sheet saved to: {OUTPUT_FILE}")
display(sheet)

Output hidden; open in https://colab.research.google.com to view.

In [30]:
from pathlib import Path
import shutil
import pandas as pd
from tqdm.auto import tqdm

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
SOURCE_MANIFEST = EXPERIMENT_DIR / "okra_manifest_after_distance_2_50_groups.csv"
DRIVE_IMAGE_DIR = EXPERIMENT_DIR / "okra_images_final"
DRIVE_MANIFEST = EXPERIMENT_DIR / "okra_final_2799_manifest_drive_paths.csv"

if not SOURCE_MANIFEST.exists():
    raise FileNotFoundError(f"Manifest not found: {SOURCE_MANIFEST}")

df = pd.read_csv(SOURCE_MANIFEST)
required = {"image_path", "label"}
if not required.issubset(df.columns):
    raise ValueError(f"Manifest must include columns: {sorted(required)}")

DRIVE_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

new_paths = []
missing_sources = []
copy_errors = []

for row in tqdm(df.itertuples(index=False), total=len(df), desc="Saving Okra images to Drive"):
    src = Path(str(row.image_path))
    label = str(row.label)

    if not src.exists():
        missing_sources.append(str(src))
        new_paths.append(None)
        continue

    # Keep class directories and original filenames to avoid name collisions.
    dst = DRIVE_IMAGE_DIR / label / src.name
    dst.parent.mkdir(parents=True, exist_ok=True)

    try:
        if not dst.exists() or dst.stat().st_size != src.stat().st_size:
            shutil.copy2(src, dst)
        new_paths.append(str(dst))
    except Exception as exc:
        copy_errors.append({"source": str(src), "error": str(exc)})
        new_paths.append(None)

if missing_sources or copy_errors:
    print("Copy incomplete; manifest not written.")
    print("Missing source files:", len(missing_sources))
    print("Copy errors:", len(copy_errors))
    if missing_sources:
        print("Examples:", missing_sources[:5])
    if copy_errors:
        print("Examples:", copy_errors[:5])
    raise RuntimeError("Resolve the reported copy problems and rerun this cell.")

saved_df = df.copy()
saved_df["original_colab_path"] = saved_df["image_path"].astype(str)
saved_df["image_path"] = new_paths

# Verify all Drive copies exist and row count is unchanged.
not_found = [p for p in saved_df["image_path"] if not Path(p).is_file()]
if not_found:
    raise RuntimeError(f"{len(not_found)} copied files are missing; manifest not saved.")

saved_df.to_csv(DRIVE_MANIFEST, index=False)

print("Images copied and verified:", len(saved_df))
print("Drive image folder:", DRIVE_IMAGE_DIR)
print("Drive manifest:", DRIVE_MANIFEST)
print("\nSaved images by class:")
print(saved_df["label"].value_counts().sort_index().to_string())

Saving Okra images to Drive:   0%|          | 0/2799 [00:00<?, ?it/s]

Images copied and verified: 2799
Drive image folder: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_images_final
Drive manifest: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_final_2799_manifest_drive_paths.csv

Saved images by class:
label
Healthy_Leaf                693
Insects_Eaten_Leaf          575
Leaf_Curl                   343
Leaf_Spot                   302
Yellow_Vein_Mosaic_Virus    886
